# 主力片段与全合约价格的复权

按 `trading_date` 划分训练期与测试期。训练期使用本期换约信息做双向复权，首尾实际主力分钟保持原价；测试期以本期第一个实际主力分钟为固定起点，只累计之后的换约偏移。训练／测试分界不作换约补偿。

同一分钟全部真实合约的 OHLC 共用主力价格算出的 log 偏移；成交量、成交额与持仓量保留原值。CU／RB 的全合约分钟长表分别保存在本方法 `c01_MainContinuousAdjustment_DemoData/`，下游用 `is_main` 选取主力，用 `segment_id` 定位片段。


## 1. 输入与时间范围

复权区间就是用来计算复权偏移的交易日范围。本演示请求 2010-01-01 至 2026-09-30，两端包含；首个已识别主力交易日为 2010-01-04，只用该日至请求终点的主力分钟行情计算偏移。

每日主力身份来自上游 对应品种的主力识别 demo（例如 [RB 每日映射](../b01_MainContractSelection/c01_MainContractSelection_DemoData/c01_MainContractSelection_RB_20100101-20260930_ratio1.10_demo.parquet)），先核对生成身份、代码和实际内容摘要。分钟行情与 Session 覆盖来自正式 silver 的 `fact_futures_minute`、`dim_futures_bar_calendar`，契约使用 `config/data_contracts.py`，湖根使用 `settings.futures_lake_root`。

| 范围 | 起点 | 终点 | 端点 |
|---|---|---|---|
| 每日映射读取区间 | 2010-01-01 | 2026-09-30 | 两端包含 |
| 分钟行情读取交易日区间 | 2010-01-04 | 2026-09-30 | 两端包含 |
| 复权交易日区间 | 2010-01-04 | 2026-09-30 | 两端包含 |

时区为 `Asia/Shanghai`。分钟归属直接使用 silver 的 `trading_date`；夜盘可落在前一个自然日，不用自然日期或固定钟点推断。读取前按交易所、品种、年、月裁剪，再按交易日过滤。实际分钟首末 `bar_at` 在 demo 中登记。训练期为 `trading_date < demo_test_start_date`，测试期为 `trading_date >= demo_test_start_date`；本演示分界为 2025-01-01。训练首尾与测试首个实际主力分钟共享零偏移。正价格保持原价，零值的派生字段仍为空。

检查范围遵循[上游契约与研究校验](../../AGENTS.md#上游契约与研究校验)。正式 silver 已保证的主键、Schema／metadata 及业务质量不在研究层重复校验；读取仍使用 PyArrow Dataset、权威 Schema、分区过滤和统一 Arrow 转换入口。主力 demo 属于研究产物，仍用其局部 Schema 统一表示，并核对生成身份和实际内容摘要。

本方法需要确认每日主力与分钟、Session 的连接关系，研究范围内的主力身份和所需分钟是否可用，片段是否有实际行情及换约所需的首开、末收 log 是否可用，以及实际参与 log 运算的价格是否为正。Session 与主力关联后形成的期望分钟网格、偏移广播后的行数和时间键、首尾锚定、同分钟合约价差、OHLC 共同偏移、数量保留及换约残差均属于本方法需要检查的条件或结果性质，不能由上游 Schema 一致推定。

训练偏移只使用训练期内部的换约；测试偏移只累计测试期内部已经发生的换约。训练／测试分界不计算换约缺口，分界价差保留。

In [1]:
import hashlib
import json
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()
for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "04_Research"))
        break
else:
    raise RuntimeError("未找到项目根目录")

print(sys.executable)
import pyarrow.parquet as pq
from IPython.display import display
from a01_Methods.b01_MainContractSelection.c01_MainContractSelection import (
    MAIN_CONTRACT_DAILY_SCHEMA,
)

E:\anaconda3\envs\latitude_env_v2\python.exe


## 2. 连续片段规则

每日映射提供完整交易日网格，按交易日升序处理。在同一个交易所和品种内，相邻交易日主力合约相同，属于同一片段；主力变化时开启新片段。周末和节假日不按自然日间隔切断片段。

未识别日保留记录，`segment_id` 为空，并切断前后已识别片段。同一合约离开主力后再次出现，得到新的片段编号。编号在本次复权区间内按时间顺序从 1 开始，不以合约代码排序或合并。

每日片段表保留交易日、合约、片段身份及片段起止交易日；片段汇总表每个片段一行。`is_final_segment=True` 表示这一行对应选定日期范围内、按时间最后的一段主力行情。最后片段也参与间隔均摊，只有序列首尾固定为原价；实际合约与起止日见下方片段汇总。

完整复权还要求区间首尾具备主力身份、区间内无未识别日；分段函数保留缺口供核查，复权函数拒绝不满足这些条件的输入。

## 3. log 域共同偏移

对正的有限价格先取自然对数：$x_{c,f,i}=\ln P_{c,f,i}$。$c$ 表示真实合约，$f$ 为 open、high、low、close，$i$ 为实际主力分钟网格位置。正式 silver 已保证非空价格有限；本方法仅对正值取 log。原始零值保留在 `raw_*`，对应 `log_*`、`adjusted_log_*` 与 `adjusted_*` 置空；原始缺失也保留为空。负价格停止复权计算。

主力片段按时间编号为 $1,\ldots,K$。从片段 $k$ 切换至 $k+1$ 的 log 缺口为：

$$
g_k=\ln O^{\mathrm{first}}_{k+1}-\ln C^{\mathrm{last}}_k.
$$

锚点取前一片段最后一个实际分钟的收盘和后一片段首个实际分钟的开盘。首末位置以读到的真实分钟行情确定，不要求与理论 Session 网格端点重合。换约锚点的原价缺失或为零、导致 log 不可用时停止计算。夜盘归属与片段分钟边界由正式交易日字段确定。

### 训练期：双向复权

只对训练内部换约计算缺口。对片段 $k$，累计后续缺口 $B_k=\sum_{j=k}^{K-1}g_j$，最后片段 $B_K=0$。记累计换月差值 $G=B_1$，实际主力分钟共 $N$ 条，位置 $i=0,\ldots,N-1$。当 $N\ge2$ 时，将 $G$ 等额分摊到全部 $N-1$ 个相邻间隔，每个间隔分摊 $q=G/(N-1)$；换月衔接和最后片段内的间隔都参与，按行序等分，不按经过的自然时间加权。

构造从 $G$ 线性变化到 0 的线性回扣 $D_i$，以累计偏移减去它：

$$
D_i=G\left(1-\frac{i}{N-1}\right),\qquad
a_i=B_{k(i)}-D_i,
$$

$$
x^{\mathrm{adj}}_{c,f,i}=x_{c,f,i}+a_i,\qquad
P^{\mathrm{adj}}_{c,f,i}=\exp(x^{\mathrm{adj}}_{c,f,i}).
$$

首行 $a_0=G-G=0$，末行 $a_{N-1}=0-0=0$，因此首尾正价格保持原价；零价格的派生字段仍为空。单片段时 $G=0$ 且全部偏移为零；只有一条分钟时没有间隔，直接取零偏移。实现时零偏移位置对 log 可用的字段直接保留原始 OHLC 数值，避免 log/exp 往返的浮点舍入改变锚定价格。

同一分钟全部合约、全部 OHLC 字段共用一个 $a_i$，因此 log 可用字段在同一时点的 log 价差保持不变。数量字段保留原值，不取 log、不参与偏移。普通相邻分钟的 log 价格变化增加 $q$；换月处先减去原始 log 缺口，再增加同样的 $q$，所以换约后首开减换约前末收等于 $q$。这一数值仅由训练期决定，$N$ 为训练实际主力分钟数。

### 测试期：固定起点的前复权

测试首个实际主力分钟偏移为零。测试第 $k$ 个片段的偏移为 $F_k=-\sum_{j=1}^{k-1}g_j$；片段内偏移不变，线性回扣为零。测试内部换约后首开与换约前末收的复权 log 差为零。测试末端不影响此前偏移，跨训练／测试分界不作换约补偿。

`backward_log_shift` 只记录训练累计后续缺口，`forward_log_shift` 只记录测试累计过去缺口的负值。另一期间的对应字段为零。共同偏移为 `backward_log_shift + forward_log_shift - linear_log_deduction`。

分钟缺失保留为诊断，不补价格、不改主力、不补偏移。日历 Session 的期望网格为 `(session_start_at, session_end_at]`，仅为主力构造该网格并与实际分钟对照。理论网格无行情的计数不等同于漏采认定，Session 的开市证据状态沿用上游日历。区间内缺失不另改主力片段编号；线性分摊仅按实际主力分钟行计数。片段首条实际分钟之前或末条实际分钟之后的理论网格缺口也只记录诊断，不阻止复权。存在无主力身份日或整段无行情时停止计算；实际换约锚点的 log 不可用时也停止，不向片段内部寻找替代价格。同一主力时点只能有一行。日级映射与偏移广播的连接用 `many_to_one` 约束关系；本方法生成的主力网格存在时间重叠时停止计算。

各合约的分钟行情可以不完全对应。例如，10:01 其他合约有行情，但主力合约没有行情，就取不到该分钟的复权偏移。

这些行情仍保留原价和 log 价；偏移、复权后 log 价和复权后价格留空，标记为 `no_main_offset`。这些时间点单独列出，不借用前后分钟的偏移。

In [2]:
from __future__ import annotations

from datetime import date
import numpy as np
import pandas as pd
import pyarrow as pa

# 正式分钟行情的权威 Schema；研究层以它为基础定义派生字段。
from config.data_contracts import FUTURES_MINUTE_SCHEMA

MAIN_CONTINUOUS_ADJUSTMENT_VERSION = "0.5.0"
ADJUSTMENT_TIMEZONE = "Asia/Shanghai"

# 以下 Schema 只定义表结构，不执行读取、复权或数据校验。
# nullable=False 表示字段不可为空；未指定时默认允许为空。
# Arrow metadata 的键和值均使用 bytes，中文内容通过 encode() 转换。

# 每个目标交易日一行：将每日主力映射扩展为主力片段信息。
# 相邻交易日主力不变则属于同一片段；主力变化时开始新片段。
MAIN_SEGMENT_DAILY_SCHEMA = pa.schema([
    pa.field("exchange_code", pa.string(), nullable=False),       # 交易所代码
    pa.field("underlying_code", pa.string(), nullable=False),     # 品种代码
    pa.field("trading_date", pa.date32(), nullable=False),        # 主力生效交易日
    pa.field("signal_trading_date", pa.date32()),                 # 主力选择使用的信息所属交易日
    pa.field("main_contract_code", pa.string()),                  # 当日主力；未识别时为空
    pa.field("segment_id", pa.int32()),                          # 片段编号；未识别日为空
    pa.field("segment_start_date", pa.date32()),                  # 当前片段在选定范围内的首交易日
    pa.field("segment_end_date", pa.date32()),                    # 当前片段在选定范围内的末交易日
    pa.field("selection_status", pa.string(), nullable=False),   # 沿用上游主力选择状态
    pa.field("is_roll", pa.bool_()),                             # 沿用上游换约标记
], metadata={
    b"method_name": b"main_continuous_adjustment_segmentation",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"grain_zh": "每个目标交易日一行，未识别日的片段编号为空".encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
})

# 每个主力片段一行：汇总该片段的合约、起止日期和交易日数量。
# 同一合约离开主力后再次成为主力，属于另一个片段。
MAIN_SEGMENT_SUMMARY_SCHEMA = pa.schema([
    pa.field("exchange_code", pa.string(), nullable=False),
    pa.field("underlying_code", pa.string(), nullable=False),
    pa.field("segment_id", pa.int32(), nullable=False),
    pa.field("main_contract_code", pa.string(), nullable=False),
    pa.field("segment_start_date", pa.date32(), nullable=False),
    pa.field("segment_end_date", pa.date32(), nullable=False),
    pa.field("trading_day_count", pa.int32(), nullable=False),    # 片段包含的交易日数量
    pa.field("is_final_segment", pa.bool_(), nullable=False),     # 是否为本次计算范围内的最后一个片段
], metadata={
    b"method_name": b"main_continuous_adjustment_segmentation",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"grain_zh": "每个连续已识别主力片段一行".encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
})

# OHLC 参与 log 变换和价格复权。
OHLC_FIELDS = ("open", "high", "low", "close")

# 数量字段保留各真实合约的原值，不取 log，也不参与价格偏移。
QUANTITY_FIELDS = ("volume", "money", "open_interest")

# 全合约分钟行情的 log 输入表：
# 保留原始字段，将 OHLC 改名为 raw_*，增加 log_* 和主力片段信息。
ALL_CONTRACT_LOG_MINUTE_SCHEMA = pa.schema([

    # 原始 OHLC 沿用上游类型和可空性；其余字段直接沿用上游字段定义。
    *(pa.field(f"raw_{field.name}", field.type, nullable=field.nullable)
      if field.name in OHLC_FIELDS else field for field in FUTURES_MINUTE_SCHEMA),

    # 正价格的自然对数；零价格和缺失价格的 log 字段为空。
    *(pa.field(f"log_{name}", pa.float64()) for name in OHLC_FIELDS),

    pa.field("main_contract_code", pa.string()),                  # 该交易日的主力合约
    pa.field("segment_id", pa.int32()),                          # 该交易日所属的主力片段
    pa.field("is_main", pa.bool_(), nullable=False),              # 本行真实合约是否为当日主力
], metadata={
    b"method_name": b"main_continuous_adjustment_log_input",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",
    b"price_domain": b"raw_* original; log_* natural logarithm",
})

# 每个实际主力分钟一行：保存由主力序列计算出的共同 log 偏移。
# 下游按 trading_date、bar_at 将偏移连接到同一时点的全部真实合约。
MINUTE_LOG_OFFSET_SCHEMA = pa.schema([
    FUTURES_MINUTE_SCHEMA.field("trading_date"),                  # 直接沿用上游交易日定义
    FUTURES_MINUTE_SCHEMA.field("bar_at"),                        # 直接沿用上游带时区分钟时间戳
    pa.field("main_contract_code", pa.string(), nullable=False),
    pa.field("segment_id", pa.int32(), nullable=False),

    # B：当前片段之后所有换约 log 价差的累计值，最后片段为 0。
    pa.field("backward_log_shift", pa.float64(), nullable=False),

    # F：测试期累计过去换约缺口的负值；训练期为 0。
    pa.field("forward_log_shift", pa.float64(), nullable=False),

    # D：训练期累计价差 G 线性递减至 0；测试期为 0。
    # 分摊依据是实际主力分钟的顺序，不按经过的自然时间加权。
    pa.field("linear_log_deduction", pa.float64(), nullable=False),

    # A = B + F - D：最终应用于该分钟全部合约 OHLC 的共同偏移。
    pa.field("log_adjustment", pa.float64(), nullable=False),
], metadata={
    b"method_name": b"main_continuous_adjustment_offset",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
})

# 最终全合约复权长表：在 log 输入表上增加偏移、状态和复权价格。
# 同时保留四种价格表示，供下游按分析需要选择。
ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA = pa.schema([
    # 包含 raw_*、log_*、数量字段、合约身份、主力身份和片段编号。
    *ALL_CONTRACT_LOG_MINUTE_SCHEMA,

    # 同时点没有实际主力分钟时，无法取得共同偏移，这三个字段允许为空。
    pa.field("backward_log_shift", pa.float64()),
    pa.field("forward_log_shift", pa.float64()),
    pa.field("linear_log_deduction", pa.float64()),
    pa.field("log_adjustment", pa.float64()),

    # adjusted 表示取得偏移；no_main_offset 表示该时点没有可用主力偏移。
    pa.field("adjustment_status", pa.string(), nullable=False),

    # 复权后的自然对数：adjusted_log_* = log_* + log_adjustment。
    *(pa.field(f"adjusted_log_{name}", pa.float64()) for name in OHLC_FIELDS),

    # 复权后的价格：adjusted_* = exp(adjusted_log_*)。
    # 零偏移且原价为正时，计算实现直接复制原价，避免 log/exp 往返误差。
    *(pa.field(f"adjusted_{name}", pa.float64()) for name in OHLC_FIELDS),
], metadata={
    b"method_name": b"main_continuous_adjustment",
    b"method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION.encode(),
    b"grain_zh": "每个真实合约的一分钟行情一行".encode(),
    b"primary_key": b"contract_code,bar_at",
    b"timezone": ADJUSTMENT_TIMEZONE.encode(),
    b"interval_closed": b"both",

    # 四种价格表示：原价、原始 log、复权 log、复权价格。
    b"price_domains": b"raw_* original; log_* natural log; adjusted_log_* adjusted natural log; adjusted_* price",

    # 换约价差取新片段首个实际分钟的 log open 减旧片段末分钟的 log close。
    # 全区间 N 个实际主力分钟形成 N-1 个间隔，换约间隔也参与分摊。
    b"offset_rule": b"split by trading_date; train B=reverse cumulative within-train gaps,F=0,D=linspace(G,0,N_train),first/last A=0; test B=D=0,F=-cumulative within-test past gaps,first A=0; A=B+F-D; no split-boundary compensation",

    # 没有主力偏移时，复权字段为空；保留原价、原始 log 和数量，不填补偏移。
    b"missing_offset_rule": b"no_main_offset: adjusted values null; raw/log/quantities preserved; no filling",
})

In [3]:
def build_main_contract_segments(
    main_contract_daily_df: pd.DataFrame,
    *,
    exchange_code: str,
    underlying_code: str,
    adjustment_start_date: str | date | None = None,
    adjustment_end_date: str | date | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """按连续交易日划分主力片段，保留未识别日。

    相邻记录中的主力合约不变，则属于同一片段。
    主力变化或中间出现未识别日后，下一次识别主力时开启新片段。
    周末、节假日没有交易日记录，不会因此拆分片段。

    Args:
        main_contract_daily_df: 已核对的完整每日映射，日期采用局部 Schema 类型。
        exchange_code: 目标交易所代码，如 "XSGE"。
        underlying_code: 目标品种代码，如 "RB"。
        adjustment_start_date: 复权区间起点，含当日；None 使用映射首日。
        adjustment_end_date: 复权区间终点，含当日；None 使用映射末日。

    Returns:
        (main_segment_daily_df, main_segment_summary_df)：
        每日片段表保留全部目标交易日，未识别日的片段编号为空；
        片段汇总表只包含已识别的主力片段。
    """

    # 1. 选择目标交易所和品种，保留划分片段及追溯主力选择所需的字段。
    # 按交易日排序，使后面的 shift() 比较当前记录与上一交易日记录。
    main_segment_daily_df = main_contract_daily_df.loc[
        (main_contract_daily_df["exchange_code"] == exchange_code)
        & (main_contract_daily_df["underlying_code"] == underlying_code),
        [
            "exchange_code", "underlying_code", "trading_date",
            "signal_trading_date", "main_contract_code",
            "selection_status", "is_roll",
        ],
    ].sort_values("trading_date").reset_index(drop=True)

    if main_segment_daily_df.empty:
        raise ValueError("目标品种没有每日主力映射")

    # 将传入的日期统一为 date；未指定时，使用目标品种映射的首末交易日。
    # 此处使用映射日期范围，不自动跳过起点处的未识别日。
    adjustment_start_date = (
        pd.Timestamp(adjustment_start_date).date()
        if adjustment_start_date is not None
        else main_segment_daily_df["trading_date"].min()
    )
    adjustment_end_date = (
        pd.Timestamp(adjustment_end_date).date()
        if adjustment_end_date is not None
        else main_segment_daily_df["trading_date"].max()
    )

    if adjustment_start_date > adjustment_end_date:
        raise ValueError("复权区间起点不能晚于终点")

    # 截取包含两端的复权区间，再在该区间内重新编号片段。
    # 因此，片段起止日期和“最后片段”均相对于本次计算范围定义。
    main_segment_daily_df = main_segment_daily_df.loc[
        main_segment_daily_df["trading_date"].between(
            adjustment_start_date, adjustment_end_date
        )
    ].reset_index(drop=True)

    if main_segment_daily_df.empty:
        raise ValueError("复权区间内没有每日主力映射")

    # 2. 标记已识别主力的交易日；未识别日保留记录，但不分配片段编号。
    identified_main_mask = main_segment_daily_df["main_contract_code"].notna()

    # 已识别且主力与前一行不同，则开启新片段。
    # 首行或前一行主力为空时，比较可能返回空值，fillna(True) 将其视为新片段。
    # 最外层 identified_main_mask 保证未识别日本身不会开启片段。
    segment_start_mask = (
        identified_main_mask
        & main_segment_daily_df["main_contract_code"].ne(
            main_segment_daily_df["main_contract_code"].shift()
        ).fillna(True)
    )

    # 起点标记 True/False 转为 1/0，累计后得到从 1 开始的片段编号。
    # where() 将未识别日的编号置空；Int32 是支持空值的 Pandas 整数类型。
    # 例如 A、A、空、A、B，对应编号 1、1、空、2、3。
    main_segment_daily_df["segment_id"] = (
        segment_start_mask.astype("int32")
        .cumsum()
        .where(identified_main_mask)
        .astype("Int32")
    )

    # 3. 仅汇总已识别日，每个片段生成一行。
    # 按 segment_id 分组，使同一合约重复出现的不同片段保持独立。
    main_segment_summary_df = (
        main_segment_daily_df.loc[identified_main_mask]
        .groupby("segment_id", as_index=False, sort=True)
        .agg(
            # 每个片段内的交易所、品种和主力合约相同，取首行即可。
            exchange_code=("exchange_code", "first"),
            underlying_code=("underlying_code", "first"),
            main_contract_code=("main_contract_code", "first"),

            # 起止日期取片段实际包含的交易日，不使用合约上市、退市日期。
            segment_start_date=("trading_date", "min"),
            segment_end_date=("trading_date", "max"),

            # 统计交易日记录数量，不计算经过的自然日数量。
            trading_day_count=("trading_date", "size"),
        )
    )

    # 编号最大的已识别片段是本次范围内的最后片段。
    # 如果全部日期都未识别，片段汇总表保持为空。
    main_segment_summary_df["is_final_segment"] = (
        main_segment_summary_df["segment_id"].eq(
            main_segment_summary_df["segment_id"].max()
        )
    )

    # 将每个片段的起止日期补回每日表。
    # 左连接保留未识别日；这些行的片段编号和片段起止日期仍为空。
    # many_to_one 约束本方法生成的连接：多个交易日只能对应一条片段汇总。
    main_segment_daily_df = main_segment_daily_df.merge(
        main_segment_summary_df[
            ["segment_id", "segment_start_date", "segment_end_date"]
        ],
        on="segment_id",
        how="left",
        validate="many_to_one",
    )

    # 4. 按局部输出 Schema 固定两张派生表的字段类型和可空性。
    # preserve_index=False 避免将 DataFrame 行索引写入 Arrow 表。
    # replace_schema_metadata 使用本方法定义的表级说明，替换转换产生的 metadata。
    main_segment_daily_table = pa.Table.from_pandas(
        main_segment_daily_df,
        schema=MAIN_SEGMENT_DAILY_SCHEMA,
        preserve_index=False,
    ).replace_schema_metadata(MAIN_SEGMENT_DAILY_SCHEMA.metadata)

    main_segment_summary_table = pa.Table.from_pandas(
        main_segment_summary_df,
        schema=MAIN_SEGMENT_SUMMARY_SCHEMA,
        preserve_index=False,
    ).replace_schema_metadata(MAIN_SEGMENT_SUMMARY_SCHEMA.metadata)

    # 转回采用 Arrow dtype 的 DataFrame，供后续分钟行情连接和复权计算使用。
    return (
        main_segment_daily_table.to_pandas(types_mapper=pd.ArrowDtype),
        main_segment_summary_table.to_pandas(types_mapper=pd.ArrowDtype),
    )

In [4]:
def prepare_main_contract_minutes(
    futures_minute_df: pd.DataFrame,
    main_segment_daily_df: pd.DataFrame,
    bar_calendar_df: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    """关联主力片段并取 log，保留原价、数量与缺失证据。

    Args:
        futures_minute_df: 按正式契约读取的目标品种全合约分钟行情，信任上游质量。
        main_segment_daily_df: 同一范围、单一品种的每日主力片段表。
        bar_calendar_df: 同范围的正式 1m Session 日历，含期望和实测条数。

    Returns:
        全合约 log 长表、主力 log 分钟表、覆盖及缺失诊断字典。
    """
    daily_df = main_segment_daily_df.sort_values("trading_date").reset_index(drop=True)
    if daily_df.empty:
        raise ValueError("每日主力片段为空")
    identities_df = daily_df[["exchange_code", "underlying_code"]].drop_duplicates()
    if len(identities_df) != 1:
        raise ValueError("一次分钟准备只能处理一个交易所和品种")
    exchange_code, underlying_code = identities_df.iloc[0]
    start_date, end_date = daily_df["trading_date"].min(), daily_df["trading_date"].max()
    minute_df = futures_minute_df.loc[
        futures_minute_df["exchange_code"].eq(exchange_code)
        & futures_minute_df["underlying_code"].eq(underlying_code)
        & futures_minute_df["trading_date"].between(start_date, end_date)
    ].copy()
    if minute_df.empty:
        raise ValueError("复权区间没有分钟行情")

    # 只按 silver trading_date 关联，many_to_one 防止日级映射放大分钟行数。
    all_contract_log_minute_df = minute_df.rename(
        columns={name: f"raw_{name}" for name in OHLC_FIELDS}
    ).merge(daily_df[["trading_date", "main_contract_code", "segment_id"]],
            on="trading_date", how="left", validate="many_to_one", indicator=True)
    if not all_contract_log_minute_df["_merge"].eq("both").all():
        raise ValueError("分钟交易日不在每日主力映射中")
    all_contract_log_minute_df = all_contract_log_minute_df.drop(columns="_merge")
    all_contract_log_minute_df["is_main"] = all_contract_log_minute_df["contract_code"].eq(
        all_contract_log_minute_df["main_contract_code"]
    ).fillna(False)
    for name in OHLC_FIELDS:
        prices = all_contract_log_minute_df[f"raw_{name}"].to_numpy(dtype=float, na_value=np.nan)
        present = all_contract_log_minute_df[f"raw_{name}"].notna().to_numpy(dtype=bool)
        invalid = present & (prices < 0)
        if invalid.any():
            sample_df = all_contract_log_minute_df.loc[invalid, ["contract_code", "bar_at", f"raw_{name}"]].head(5)
            raise ValueError(f"{name} 含 {int(invalid.sum())} 个负价格：\n{sample_df}")
        log_prices = np.full(len(prices), np.nan)
        # 原始零值保留；只有正值参与 log，零值的派生价格字段为空。
        positive = present & (prices > 0)
        log_prices[positive] = np.log(prices[positive])
        all_contract_log_minute_df[f"log_{name}"] = pd.array(log_prices, dtype=pd.ArrowDtype(pa.float64()))
    all_contract_log_minute_df = pa.Table.from_pandas(
        all_contract_log_minute_df, schema=ALL_CONTRACT_LOG_MINUTE_SCHEMA, preserve_index=False,
    ).to_pandas(types_mapper=pd.ArrowDtype).sort_values(["bar_at", "contract_code"]).reset_index(drop=True)
    main_log_minute_df = all_contract_log_minute_df.loc[all_contract_log_minute_df["is_main"]].reset_index(drop=True)
    if main_log_minute_df["bar_at"].duplicated().any():
        raise ValueError("主力分钟时间重叠，不能建立唯一共同偏移")

    # 日历已提交的覆盖证明直接展示；仅构造选中主力的期望网格定位本算法缺口。
    session_coverage_df = bar_calendar_df.loc[
        bar_calendar_df["bar_frequency"].eq("1m")
        & bar_calendar_df["exchange_code"].eq(exchange_code)
        & bar_calendar_df["underlying_code"].eq(underlying_code)
        & bar_calendar_df["trading_date"].between(start_date, end_date),
        ["contract_code", "trading_date", "session_number", "session_start_at", "session_end_at",
         "schedule_status", "schedule_signal_reason", "is_fetch_required", "is_fetch_completed",
         "expected_bar_count", "actual_bar_count", "missing_bar_count", "quality_status", "quality_reason"],
    ].merge(daily_df[["trading_date", "main_contract_code", "segment_id"]],
            on="trading_date", how="left", validate="many_to_one")
    session_coverage_df["is_main"] = session_coverage_df["contract_code"].eq(session_coverage_df["main_contract_code"]).fillna(False)
    main_sessions_df = session_coverage_df.loc[session_coverage_df["is_main"] & session_coverage_df["is_fetch_required"]]
    missing_calendar_dates = daily_df.loc[daily_df["main_contract_code"].notna() & ~daily_df["trading_date"].isin(main_sessions_df["trading_date"])]
    if not missing_calendar_dates.empty:
        raise ValueError(f"主力缺少 required Session 日历：{missing_calendar_dates['trading_date'].tolist()}")
    expected_parts = []
    for session in main_sessions_df.itertuples(index=False):
        timestamps = pd.date_range(session.session_start_at, session.session_end_at, freq="min", inclusive="right")
        expected_parts.append(pd.DataFrame({
            "trading_date": session.trading_date, "bar_at": timestamps,
            "contract_code": session.contract_code, "segment_id": session.segment_id,
            "session_number": session.session_number,
        }))
    if not expected_parts:
        raise ValueError("没有可识别主力的期望分钟网格")
    expected_main_minute_df = pd.concat(expected_parts, ignore_index=True)
    expected_main_minute_df["bar_at"] = pd.array(expected_main_minute_df["bar_at"], dtype=pd.ArrowDtype(FUTURES_MINUTE_SCHEMA.field("bar_at").type))
    expected_main_minute_df["trading_date"] = pd.array(expected_main_minute_df["trading_date"], dtype=pd.ArrowDtype(pa.date32()))
    if expected_main_minute_df.duplicated(["contract_code", "bar_at"]).any() or expected_main_minute_df["bar_at"].duplicated().any():
        raise ValueError("生成的主力期望分钟网格存在 Session 重叠")
    main_grid_comparison_df = expected_main_minute_df.merge(
        main_log_minute_df[["contract_code", "bar_at", "trading_date"]],
        on=["contract_code", "bar_at", "trading_date"], how="outer", validate="one_to_one", indicator=True,
    )
    if main_grid_comparison_df["_merge"].eq("right_only").any():
        raise ValueError("主力分钟落在本范围日历期望网格之外")
    missing_main_minute_df = main_grid_comparison_df.loc[main_grid_comparison_df["_merge"].eq("left_only")].drop(columns="_merge").sort_values("bar_at").reset_index(drop=True)
    main_daily_coverage_df = daily_df[["trading_date", "main_contract_code", "segment_id"]].copy()
    for column, source_df in (("expected_minute_count", expected_main_minute_df),
                              ("observed_minute_count", main_log_minute_df),
                              ("missing_minute_count", missing_main_minute_df)):
        counts = source_df.groupby("trading_date").size()
        main_daily_coverage_df[column] = main_daily_coverage_df["trading_date"].map(counts).fillna(0).astype(pd.ArrowDtype(pa.int32()))
    all_minute_times_df = all_contract_log_minute_df[["trading_date", "bar_at"]].drop_duplicates()
    unsupported_times_df = all_minute_times_df.merge(main_log_minute_df[["trading_date", "bar_at"]],
        on=["trading_date", "bar_at"], how="left", validate="one_to_one", indicator=True)
    unsupported_times_df = unsupported_times_df.loc[unsupported_times_df["_merge"].eq("left_only")].drop(columns="_merge")
    diagnostics = {
        "session_coverage_df": session_coverage_df,
        "missing_main_minute_df": missing_main_minute_df,
        "main_daily_coverage_df": main_daily_coverage_df,
        "unresolved_main_daily_df": daily_df.loc[daily_df["main_contract_code"].isna()],
        "all_contract_times_without_main_df": unsupported_times_df,
    }
    return all_contract_log_minute_df, main_log_minute_df, diagnostics


In [5]:
def apply_minute_log_offsets(
    all_contract_log_minute_df: pd.DataFrame,
    minute_log_offset_df: pd.DataFrame,
) -> pd.DataFrame:
    """将已由完整复权区间确定的偏移广播到一批全合约行情。

    分批只改变内存占用，不重新初始化片段或线性回扣。
    """
    # 以 trading_date + bar_at 广播；无主力偏移的合约时点保留空值，绝不补偏移。
    broadcast_columns = ["trading_date", "bar_at", "backward_log_shift", "forward_log_shift", "linear_log_deduction", "log_adjustment"]
    adjusted_all_contract_minute_df = all_contract_log_minute_df.merge(
        minute_log_offset_df[broadcast_columns], on=["trading_date", "bar_at"],
        how="left", validate="many_to_one",
    )
    broadcast_offsets = adjusted_all_contract_minute_df["log_adjustment"].to_numpy(dtype=float, na_value=np.nan)
    adjusted_all_contract_minute_df["adjustment_status"] = pd.array(
        np.where(np.isfinite(broadcast_offsets), "adjusted", "no_main_offset"), dtype=pd.ArrowDtype(pa.string()))
    zero_offset = broadcast_offsets == 0
    for name in OHLC_FIELDS:
        original_log = adjusted_all_contract_minute_df[f"log_{name}"].to_numpy(dtype=float, na_value=np.nan)
        adjusted_log = original_log + broadcast_offsets
        with np.errstate(over="raise", invalid="raise"):
            adjusted_prices = np.exp(adjusted_log)
        # A=0 且 log 可用时复制原价；原始零值的派生价格仍为空。
        original_prices = adjusted_all_contract_minute_df[f"raw_{name}"].to_numpy(dtype=float, na_value=np.nan)
        copy_original = zero_offset & np.isfinite(original_log)
        adjusted_prices[copy_original] = original_prices[copy_original]
        if ((~np.isnan(adjusted_prices)) & (~np.isfinite(adjusted_prices) | (adjusted_prices <= 0))).any():
            raise ValueError("复权后价格发生数值溢出或下溢")
        adjusted_all_contract_minute_df[f"adjusted_log_{name}"] = pd.array(adjusted_log, dtype=pd.ArrowDtype(pa.float64()))
        adjusted_all_contract_minute_df[f"adjusted_{name}"] = pd.array(adjusted_prices, dtype=pd.ArrowDtype(pa.float64()))
    adjusted_all_contract_minute_df = pa.Table.from_pandas(
        adjusted_all_contract_minute_df, schema=ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA, preserve_index=False,
    ).to_pandas(types_mapper=pd.ArrowDtype)
    return adjusted_all_contract_minute_df


def _adjust_one_period(
    main_log_minute_df: pd.DataFrame,
    all_contract_log_minute_df: pd.DataFrame,
    main_segment_daily_df: pd.DataFrame,
    missing_main_minute_df: pd.DataFrame,
    *, adjustment_mode: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """计算一期偏移：训练双向复权，测试固定首点。

    同一时间键的全合约 OHLC 共用偏移；间隔按实际主力分钟行序计数。

    Args:
        main_log_minute_df: prepare_main_contract_minutes 返回的主力 log 行情。
        all_contract_log_minute_df: 同次准备的全合约原价与 log 长表。
        main_segment_daily_df: 固定复权范围的日级片段，须全部识别。
        missing_main_minute_df: 同次准备的期望主力分钟缺失明细。

    Returns:
        复权主力、复权全合约、逐分钟偏移、片段/换约及缺口诊断。
    """
    main_df = main_log_minute_df.sort_values("bar_at").reset_index(drop=True)
    daily_df = main_segment_daily_df.sort_values("trading_date")
    if main_df.empty or daily_df.empty or daily_df["segment_id"].isna().any():
        raise ValueError("复权范围必须有行情且每日主力全部识别")
    segment_ids = daily_df["segment_id"].drop_duplicates().tolist()
    if main_df["segment_id"].drop_duplicates().tolist() != segment_ids:
        raise ValueError("片段整段缺行情或分钟片段时间顺序错误")
    selected_all_df = all_contract_log_minute_df.loc[all_contract_log_minute_df["is_main"]].sort_values("bar_at").reset_index(drop=True)
    if not main_df.equals(selected_all_df):
        raise ValueError("主力与全合约输入不是同一次分钟准备结果")

    segment_diagnostics_df = main_df.groupby("segment_id", sort=False).agg(
        main_contract_code=("contract_code", "first"),
        first_bar_at=("bar_at", "first"), last_bar_at=("bar_at", "last"),
        first_trading_date=("trading_date", "first"), last_trading_date=("trading_date", "last"),
        minute_count=("bar_at", "size"),
    ).reset_index()
    # 首末锚点按片段实际分钟确定，理论网格缺口仅保留为诊断。
    first_rows_df = main_df.drop_duplicates("segment_id", keep="first").reset_index(drop=True)
    last_rows_df = main_df.drop_duplicates("segment_id", keep="last").reset_index(drop=True)
    old_log_close = last_rows_df["log_close"].to_numpy(dtype=float, na_value=np.nan)[:-1]
    new_log_open = first_rows_df["log_open"].to_numpy(dtype=float, na_value=np.nan)[1:]
    if not np.isfinite(old_log_close).all() or not np.isfinite(new_log_open).all():
        raise ValueError("换约首开或末收 log 锚点缺失或非有限")

    # g>0 表示新主力开盘更高；旧片段加上后续 g 的总和，以末片段为后复权锚点。
    gaps = new_log_open - old_log_close
    backward_shifts = np.r_[np.cumsum(gaps[::-1])[::-1], 0.0] if adjustment_mode == "train" else np.zeros(len(segment_ids))
    forward_shifts = -np.r_[0.0, np.cumsum(gaps)] if adjustment_mode == "test" else np.zeros(len(segment_ids))
    segment_diagnostics_df["backward_log_shift"] = pd.array(backward_shifts, dtype=pd.ArrowDtype(pa.float64()))
    segment_diagnostics_df["forward_log_shift"] = forward_shifts
    segment_diagnostics_df["adjustment_period"] = adjustment_mode
    total_log_gap = float(backward_shifts[0])
    final_segment_id = segment_ids[-1]
    main_minute_count = len(main_df)
    interval_count = main_minute_count - 1
    per_interval_log_gap = total_log_gap / interval_count if interval_count else 0.0
    # D 覆盖全部实际主力分钟；相邻 D 相差 G/(N-1)，换月间隔同样参与。
    # N=1 时必为单片段，G=0，得到唯一的零偏移。
    deductions = np.linspace(total_log_gap, 0.0, main_minute_count)
    shift_by_segment = dict(zip(segment_ids, backward_shifts))
    forward_by_segment = dict(zip(segment_ids, forward_shifts))
    offsets = (main_df["segment_id"].map(shift_by_segment).to_numpy(dtype=float)
        + main_df["segment_id"].map(forward_by_segment).to_numpy(dtype=float) - deductions)
    minute_log_offset_df = main_df[["trading_date", "bar_at", "contract_code", "segment_id"]].rename(columns={"contract_code": "main_contract_code"}).copy()
    minute_log_offset_df["backward_log_shift"] = main_df["segment_id"].map(shift_by_segment)
    minute_log_offset_df["forward_log_shift"] = main_df["segment_id"].map(forward_by_segment)
    minute_log_offset_df["linear_log_deduction"] = deductions
    minute_log_offset_df["log_adjustment"] = offsets
    minute_log_offset_df = pa.Table.from_pandas(minute_log_offset_df, schema=MINUTE_LOG_OFFSET_SCHEMA, preserve_index=False).to_pandas(types_mapper=pd.ArrowDtype)

    adjusted_all_contract_minute_df = apply_minute_log_offsets(all_contract_log_minute_df, minute_log_offset_df)
    adjusted_main_minute_df = adjusted_all_contract_minute_df.loc[adjusted_all_contract_minute_df["is_main"]].sort_values("bar_at").reset_index(drop=True)
    first_positions = np.flatnonzero(main_df["segment_id"].ne(main_df["segment_id"].shift()).fillna(True).to_numpy(dtype=bool))
    new_positions = first_positions[1:]
    old_positions = new_positions - 1
    roll_diagnostics_df = pd.DataFrame({
        "adjustment_period": adjustment_mode,
        "old_segment_id": segment_ids[:-1], "new_segment_id": segment_ids[1:],
        "old_contract_code": last_rows_df["contract_code"].tolist()[:-1],
        "new_contract_code": first_rows_df["contract_code"].tolist()[1:],
        "old_bar_at": last_rows_df["bar_at"].tolist()[:-1],
        "new_bar_at": first_rows_df["bar_at"].tolist()[1:],
        "gap_log": gaps, "old_linear_log_deduction": deductions[old_positions],
        "new_linear_log_deduction": deductions[new_positions],
        "expected_residual_log": np.full(len(gaps), per_interval_log_gap),
        "actual_residual_log": adjusted_main_minute_df["adjusted_log_open"].to_numpy(dtype=float, na_value=np.nan)[new_positions]
            - adjusted_main_minute_df["adjusted_log_close"].to_numpy(dtype=float, na_value=np.nan)[old_positions],
    })
    assert offsets[0] == 0
    if adjustment_mode == "train":
        assert offsets[-1] == 0
    assert np.allclose(roll_diagnostics_df["actual_residual_log"], roll_diagnostics_df["expected_residual_log"], atol=2e-14, rtol=1e-10)
    segment_diagnostics_df["missing_minute_count"] = segment_diagnostics_df["segment_id"].map(missing_main_minute_df.groupby("segment_id").size()).fillna(0).astype(pd.ArrowDtype(pa.int32()))
    segment_diagnostics_df["is_final_segment"] = segment_diagnostics_df["segment_id"].eq(final_segment_id)
    diagnostics = {
        "segment_diagnostics_df": segment_diagnostics_df,
        "roll_diagnostics_df": roll_diagnostics_df,
        "all_contract_times_without_offset_df": adjusted_all_contract_minute_df.loc[
            adjusted_all_contract_minute_df["log_adjustment"].isna(), ["trading_date", "bar_at"]].drop_duplicates(),
        "missing_main_minute_df": missing_main_minute_df,
        "main_minute_count": main_minute_count,
        "interval_count": interval_count,
        "per_interval_log_gap": per_interval_log_gap,
        "total_log_gap": total_log_gap,
    }
    return adjusted_main_minute_df, adjusted_all_contract_minute_df, minute_log_offset_df, diagnostics


def adjust_main_contract_minutes(
    main_log_minute_df: pd.DataFrame,
    all_contract_log_minute_df: pd.DataFrame,
    main_segment_daily_df: pd.DataFrame,
    missing_main_minute_df: pd.DataFrame,
    *, test_start_date: str | date,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict]:
    """按 trading_date 切分，训练双向复权、测试固定首个实际分钟。

    分界不补偿；interval_count、total_log_gap、per_interval_log_gap 指训练期。
    """
    boundary = pd.Timestamp(test_start_date).date()
    if main_log_minute_df.empty or main_segment_daily_df.empty:
        raise ValueError("复权范围必须有行情且每日主力全部识别")
    selected_main = all_contract_log_minute_df.loc[all_contract_log_minute_df["is_main"]].sort_values("bar_at").reset_index(drop=True)
    if not main_log_minute_df.sort_values("bar_at").reset_index(drop=True).equals(selected_main):
        raise ValueError("主力与全合约输入不是同一次分钟准备结果")
    all_parts, offset_parts, segment_parts, roll_parts, stats = [], [], [], [], {}
    for period in ("train", "test"):
        is_training = period == "train"
        daily = main_segment_daily_df.loc[(main_segment_daily_df["trading_date"] < boundary).eq(is_training)]
        period_main_df = main_log_minute_df.loc[(main_log_minute_df["trading_date"] < boundary).eq(is_training)]
        period_all_df = all_contract_log_minute_df.loc[(all_contract_log_minute_df["trading_date"] < boundary).eq(is_training)]
        period_missing_df = missing_main_minute_df.loc[(missing_main_minute_df["trading_date"] < boundary).eq(is_training)]
        if daily.empty:
            stats[period] = {"main_minute_count": 0, "interval_count": 0,
                "total_log_gap": 0.0, "per_interval_log_gap": 0.0}
            continue
        _, adjusted_all, offsets, diagnostics = _adjust_one_period(
            period_main_df, period_all_df, daily,
            period_missing_df, adjustment_mode=period)
        all_parts.append(adjusted_all); offset_parts.append(offsets)
        segment_parts.append(diagnostics["segment_diagnostics_df"])
        roll_parts.append(diagnostics["roll_diagnostics_df"])
        stats[period] = {key: diagnostics[key] for key in (
            "main_minute_count", "interval_count", "total_log_gap", "per_interval_log_gap")}
    adjusted_all = pd.concat(all_parts, ignore_index=True).sort_values(["bar_at", "contract_code"]).reset_index(drop=True)
    adjusted_main = adjusted_all.loc[adjusted_all["is_main"]].reset_index(drop=True)
    offsets = pd.concat(offset_parts, ignore_index=True).sort_values("bar_at").reset_index(drop=True)
    diagnostics = {
        "segment_diagnostics_df": pd.concat(segment_parts, ignore_index=True),
        "roll_diagnostics_df": pd.concat(roll_parts, ignore_index=True),
        "all_contract_times_without_offset_df": adjusted_all.loc[
            adjusted_all["log_adjustment"].isna(), ["trading_date", "bar_at"]].drop_duplicates(),
        "missing_main_minute_df": missing_main_minute_df,
        "test_start_date": boundary.isoformat(),
        "main_minute_count": len(adjusted_main),
        "training_minute_count": stats["train"]["main_minute_count"],
        "test_minute_count": stats["test"]["main_minute_count"],
        "interval_count": stats["train"]["interval_count"],
        "total_log_gap": stats["train"]["total_log_gap"],
        "per_interval_log_gap": stats["train"]["per_interval_log_gap"],
    }
    return adjusted_main, adjusted_all, offsets, diagnostics


In [6]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import base64
import json
import os
import pyarrow.parquet as pq


class ArrowContentDigest:
    """对规范顺序的数据按固定 65,536 行 Arrow batch 计算内容摘要。

    update 的输入须已按调用方声明的全局顺序排列；内部合并任意读取块，
    因而结果不依赖月界、Parquet row group 或调用方的分批大小。
    Schema 与每个 batch 的 IPC 表示参与摘要，demo_identity 不参与。
    """
    def __init__(self, schema: pa.Schema):
        self.schema = schema
        self.digest = hashlib.sha256(schema.serialize().to_pybytes())
        self.pending = []
        self.pending_rows = 0

    def update(self, table: pa.Table) -> None:
        table = table.select(self.schema.names).cast(self.schema).replace_schema_metadata(self.schema.metadata)
        position = 0
        while position < table.num_rows:
            take = min(65536 - self.pending_rows, table.num_rows - position)
            self.pending.append(table.slice(position, take))
            self.pending_rows += take
            position += take
            if self.pending_rows == 65536:
                canonical = pa.concat_tables(self.pending).combine_chunks()
                canonical = canonical.take(pa.array(range(canonical.num_rows), type=pa.int32()))
                batch = canonical.to_batches()[0]
                self.digest.update(batch.serialize())
                self.pending = []
                self.pending_rows = 0

    def hexdigest(self) -> str:
        digest = self.digest.copy()
        if self.pending_rows:
            canonical = pa.concat_tables(self.pending).combine_chunks()
            canonical = canonical.take(pa.array(range(canonical.num_rows), type=pa.int32()))
            batch = canonical.to_batches()[0]
            digest.update(batch.serialize())
        return digest.hexdigest()


def arrow_content_sha256(table: pa.Table | str | Path, schema: pa.Schema, sort_keys: list[tuple[str, str]]) -> str:
    """计算实际内容摘要。Table 先排序；文件须已按 sort_keys 保存。

    文件逐批读取，内存用量与全文件大小无关。调用方负责生成身份与范围匹配。
    """
    digest = ArrowContentDigest(schema)
    if isinstance(table, pa.Table):
        digest.update(table.sort_by(sort_keys))
    else:
        for batch in pq.ParquetFile(table).iter_batches(batch_size=65536, columns=schema.names):
            digest.update(pa.Table.from_batches([batch]))
    return digest.hexdigest()


def save_main_adjustment_demo(adjusted_batches, *, demo_path: str | Path, demo_identity: dict) -> dict:
    """保存一张全合约长表，接受 DataFrame 或按时间递增的 DataFrame 迭代器。

    每批内部按 bar_at、contract_code 排序；不同批次的时间不得重叠。
    分批写入后复算实际内容摘要。身份和实际内容一致时保留原文件字节及时间。
    """
    demo_path = Path(demo_path)
    schema = ALL_CONTRACT_ADJUSTED_MINUTE_SCHEMA
    output_sort_keys = [("bar_at", "ascending"), ("contract_code", "ascending")]
    staged_path = demo_path.with_name(demo_path.name + ".staging")
    if staged_path.exists():
        raise RuntimeError(f"存在未处理 staging，请先核查：{staged_path}")
    batches = [adjusted_batches] if isinstance(adjusted_batches, pd.DataFrame) else adjusted_batches
    digest = ArrowContentDigest(schema)
    row_count, previous_end = 0, None
    with pq.ParquetWriter(staged_path, schema, compression="zstd", compression_level=7) as writer:
        for adjusted_df in batches:
            table = pa.Table.from_pandas(adjusted_df, schema=schema, preserve_index=False).replace_schema_metadata(schema.metadata)
            table = table.sort_by(output_sort_keys).combine_chunks()
            if not table.num_rows:
                continue
            for field in schema:
                if not field.nullable and table[field.name].null_count:
                    raise ValueError(f"输出强制字段 {field.name} 出现空值")
            if previous_end is not None and table['bar_at'][0].as_py() <= previous_end:
                raise ValueError("输出批次时间重叠或逆序")
            previous_end = table['bar_at'][-1].as_py()
            digest.update(table)
            row_count += table.num_rows
            writer.write_table(table, row_group_size=131072)
        if not row_count:
            raise ValueError("复权输出为空")
        expected_identity = json.loads(json.dumps({**demo_identity,
            "output_rows": row_count,
            "output_schema_sha256": hashlib.sha256(schema.serialize().to_pybytes()).hexdigest(),
            "output_content_sha256": digest.hexdigest(), "output_sort_keys": output_sort_keys,
        }, ensure_ascii=False, allow_nan=False))
        installed_identity = {**expected_identity, "generated_at_utc": datetime.now(timezone.utc).isoformat()}
        output_metadata = {**schema.metadata, b"demo_identity": json.dumps(installed_identity, ensure_ascii=False, allow_nan=False, sort_keys=True).encode()}
        # 同步 Arrow 序列化 Schema，确保 Arrow 读取端恢复最终身份和时区。
        writer.add_key_value_metadata({**output_metadata,
            b"ARROW:schema": base64.b64encode(schema.with_metadata(output_metadata).serialize().to_pybytes())})
    if arrow_content_sha256(staged_path, schema, output_sort_keys) != expected_identity['output_content_sha256']:
        raise RuntimeError("staging 实际内容摘要不一致，保留现场")
    preserved_existing = False
    if demo_path.exists():
        existing_schema = pq.read_schema(demo_path)
        existing_identity = json.loads(existing_schema.metadata[b'demo_identity'])
        existing_base_schema = existing_schema.with_metadata({k: v for k, v in existing_schema.metadata.items() if k != b'demo_identity'})
        existing_content = arrow_content_sha256(demo_path, existing_base_schema, existing_identity['output_sort_keys'])
        if existing_content != existing_identity['output_content_sha256']:
            raise RuntimeError("已有 demo 实际内容与自身摘要不一致")
        existing_base_schema = existing_schema.with_metadata({k: v for k, v in existing_schema.metadata.items() if k != b'demo_identity'})
        if (existing_base_schema.equals(schema, check_metadata=True)
                and {k: v for k, v in existing_identity.items() if k != 'generated_at_utc'} == expected_identity):
            preserved_existing = True
            installed_identity = existing_identity
            staged_path.unlink()  # 本次创建并验收的事务暂存文件。
    if not preserved_existing:
        os.replace(staged_path, demo_path)
    installed_schema = pq.read_schema(demo_path)
    installed_metadata = installed_schema.metadata
    installed_base_schema = installed_schema.with_metadata({k: v for k, v in installed_metadata.items() if k != b'demo_identity'})
    if not installed_base_schema.equals(schema, check_metadata=True):
        raise RuntimeError('安装后 Schema 不一致')
    if json.loads(installed_metadata[b'demo_identity']) != installed_identity:
        raise RuntimeError("安装后生成身份不一致")
    if arrow_content_sha256(demo_path, schema, output_sort_keys) != expected_identity['output_content_sha256']:
        raise RuntimeError("安装后实际内容摘要不一致")
    return {"identity": installed_identity, "preserved_existing": preserved_existing, "file_bytes": demo_path.stat().st_size}


## 4. 铜与螺纹钢：2010-01-01 至 2026-09-30

`demo_underlying_code` 选择铜 `CU` 或螺纹钢 `RB`，各输出一个全合约复权分钟文件。文件名写明品种、请求范围、主力换约门槛 `ratio1.10` 和复权方法 `log_common`；下游通过完整文件名选择输入，原有 metadata 保存精确范围、输入与代码身份。

主力映射从首个有效交易日按当日成交量初始化；CU、RB 本次首日均为 2010-01-04。证据不足或没有正成交量候选导致的初始化前日期仍保留“未识别”。复权仅跳过开头尚未识别的日期，从首个已识别交易日起计算；中途未识别、片段首末或换约锚点缺失仍会停止计算，分钟缺失不补价。`mapping_read` 记录请求范围，`minute_read_trading_dates` 与 `adjustment_trading_dates` 记录实际计算范围。

全历史按月读取以限制内存。第一遍收集实际主力分钟，分别计算训练双向偏移和测试固定起点偏移；第二遍将这组全局偏移广播到各月全合约行情，逐批写入同一个 Parquet。月界不重置复权，输出结果与相同全区间的一次性计算口径一致。


In [7]:
demo_exchange_code = "XSGE"
demo_underlying_code = "RB"
demo_mapping_read_start_date = date(2010, 1, 1)
demo_mapping_read_end_date = date(2026, 9, 30)
demo_roll_trigger_ratio = 1.10
demo_test_start_date = date(2025, 1, 1)
# 文件名保留参数精度，最少两位小数便于辨认 1.10 门槛。
from decimal import Decimal
ratio_integer, _, ratio_fraction = format(Decimal(str(demo_roll_trigger_ratio)), "f").partition(".")
demo_ratio_token = ratio_integer + "." + ratio_fraction.ljust(2, "0")
demo_adjustment_start_date = demo_mapping_read_start_date
demo_adjustment_end_date = demo_mapping_read_end_date
demo_scope = {
    "mapping_read": {"start": demo_mapping_read_start_date.isoformat(), "end": demo_mapping_read_end_date.isoformat()},
    "minute_read_trading_dates": {"start": demo_adjustment_start_date.isoformat(), "end": demo_adjustment_end_date.isoformat()},
    "adjustment_trading_dates": {"start": demo_adjustment_start_date.isoformat(), "end": demo_adjustment_end_date.isoformat()},
    "training_trading_dates": {"start": demo_adjustment_start_date.isoformat(), "end": (pd.Timestamp(demo_test_start_date)-pd.Timedelta(days=1)).date().isoformat()},
    "test_trading_dates": {"start": demo_test_start_date.isoformat(), "end": demo_adjustment_end_date.isoformat()},
    "timezone": ADJUSTMENT_TIMEZONE, "interval_closed": "both",
}
upstream_demo_name = (f"c01_MainContractSelection_{demo_underlying_code}_"
    f"{demo_mapping_read_start_date:%Y%m%d}-{demo_mapping_read_end_date:%Y%m%d}_ratio{demo_ratio_token}_demo.parquet")
upstream_demo_path = candidate_root / "04_Research/a01_Methods/b01_MainContractSelection/c01_MainContractSelection_DemoData" / upstream_demo_name
upstream_notebook_path = upstream_demo_path.parent.parent / "c01_MainContractSelection.ipynb"
demo_scope_column_labels = {
    "start": "start（起点）", "end": "end（终点）",
    "timezone": "timezone（时区）", "interval_closed": "interval_closed（区间端点规则）",
    "mapping_read": "mapping_read（主力映射读取范围）",
    "minute_read_trading_dates": "minute_read_trading_dates（分钟读取交易日范围）",
    "adjustment_trading_dates": "adjustment_trading_dates（复权交易日范围）",
    "training_trading_dates": "training_trading_dates（训练交易日范围）",
    "test_trading_dates": "test_trading_dates（测试交易日范围）",
}
display(pd.DataFrame(demo_scope).T.rename(columns=demo_scope_column_labels, index=demo_scope_column_labels))


,start（起点）,end（终点）
mapping_read（主力映射读取范围）,2010-01-01,2026-09-30
minute_read_trading_dates（分钟读取交易日范围）,2010-01-01,2026-09-30
adjustment_trading_dates（复权交易日范围）,2010-01-01,2026-09-30
training_trading_dates（训练交易日范围）,2010-01-01,2024-12-31
test_trading_dates（测试交易日范围）,2025-01-01,2026-09-30
timezone（时区）,Asia/Shanghai,Asia/Shanghai
interval_closed（区间端点规则）,both,both


In [8]:
# 展示标签不参与算法摘要，silver 字段说明直接从权威 Schema 读取。
from a01_Methods.b01_MainContractSelection.c01_MainContractSelection import MAIN_CONTRACT_COLUMN_LABELS

ADJUSTMENT_COLUMN_LABELS = {
    **MAIN_CONTRACT_COLUMN_LABELS,
    **{
        field.name: f"{field.name}（{field.metadata[b'field_name_zh'].decode('utf-8')}）"
        for field in FUTURES_MINUTE_SCHEMA
    },
}
adjustment_derived_column_labels = {
    'segment_id': 'segment_id（连续主力片段编号）',
    'segment_start_date': 'segment_start_date（片段首交易日）',
    'segment_end_date': 'segment_end_date（片段末交易日）',
    'trading_day_count': 'trading_day_count（片段交易日数）',
    'is_final_segment': 'is_final_segment（是否最后片段）',
    'backward_log_shift': 'backward_log_shift（训练累计后续换约对数偏移）',
    'forward_log_shift': 'forward_log_shift（测试累计过去换约对数偏移）',
    'adjustment_period': 'adjustment_period（复权所属训练或测试期）',
    'training_trading_dates': 'training_trading_dates（训练交易日范围）',
    'test_trading_dates': 'test_trading_dates（测试交易日范围）',
    'linear_log_deduction': 'linear_log_deduction（线性扣除的对数偏移）',
    'log_adjustment': 'log_adjustment（共同复权对数偏移）',
    'adjustment_status': 'adjustment_status（复权状态）',
    'is_main': 'is_main（是否当日主力）',
    'first_bar_at': 'first_bar_at（片段首实际分钟时点）',
    'last_bar_at': 'last_bar_at（片段末实际分钟时点）',
    'first_trading_date': 'first_trading_date（片段首实际分钟所属交易日）',
    'last_trading_date': 'last_trading_date（片段末实际分钟所属交易日）',
    'minute_count': 'minute_count（实际主力分钟数）',
    'expected_minute_count': 'expected_minute_count（理论主力分钟数）',
    'observed_minute_count': 'observed_minute_count（实际主力分钟数）',
    'missing_minute_count': 'missing_minute_count（缺失主力分钟数）',
    'old_segment_id': 'old_segment_id（换约前片段编号）',
    'new_segment_id': 'new_segment_id（换约后片段编号）',
    'old_contract_code': 'old_contract_code（换约前合约）',
    'new_contract_code': 'new_contract_code（换约后合约）',
    'old_bar_at': 'old_bar_at（换约前末分钟时点）',
    'new_bar_at': 'new_bar_at（换约后首分钟时点）',
    'gap_log': 'gap_log（新合约首开盘减旧合约末收盘的对数价差）',
    'old_linear_log_deduction': 'old_linear_log_deduction（换约前末分钟线性扣除量）',
    'new_linear_log_deduction': 'new_linear_log_deduction（换约后首分钟线性扣除量）',
    'expected_residual_log': 'expected_residual_log（换约边界预期残余对数价差）',
    'actual_residual_log': 'actual_residual_log（换约边界实际残余对数价差）',
    'field': 'field（字段）',
    'type': 'type（Arrow 类型）',
    'nullable': 'nullable（是否允许空值）',
    'table': 'table（上游表名）',
    'rows': 'rows（输入行数）',
    'sha256': 'sha256（输入内容摘要）',
    'schema_version': 'schema_version（契约版本）',
    'producer': 'producer（生成 Notebook）',
    'generated_at_utc': 'generated_at_utc（生成时间 UTC）',
    'method_version': 'method_version（方法版本）',
    'source_sha256': 'source_sha256（算法定义摘要）',
    'content_sha256': 'content_sha256（输出内容摘要）',
    'start': 'start（起点）',
    'end': 'end（终点）',
    'timezone': 'timezone（时区）',
    'interval_closed': 'interval_closed（区间端点规则）',
    'mapping_read': 'mapping_read（主力映射读取范围）',
    'minute_read_trading_dates': 'minute_read_trading_dates（分钟读取交易日范围）',
    'adjustment_trading_dates': 'adjustment_trading_dates（复权交易日范围）',
}
ADJUSTMENT_COLUMN_LABELS.update(adjustment_derived_column_labels)
for price_name in OHLC_FIELDS:
    price_label = FUTURES_MINUTE_SCHEMA.field(price_name).metadata[b"field_name_zh"].decode("utf-8")
    for prefix, meaning in (
        ("raw_", f"原始{price_label}"), ("log_", f"原始{price_label}自然对数"),
        ("adjusted_log_", f"复权{price_label}自然对数"), ("adjusted_", f"复权{price_label}"),
    ):
        ADJUSTMENT_COLUMN_LABELS[prefix + price_name] = f"{prefix}{price_name}（{meaning}）"


In [9]:
# 清空核对标记，防止上次成功读取的变量掩盖本次失败。
verified_main_contract_daily_df = None
verified_upstream_identity = None
main_segment_daily_df = None
main_segment_summary_df = None
upstream_demo_table = pq.read_table(upstream_demo_path).combine_chunks()
upstream_metadata = upstream_demo_table.schema.metadata or {}
upstream_demo_identity = json.loads(upstream_metadata[b"demo_identity"])
expected_producer = "04_Research/a01_Methods/b01_MainContractSelection/c01_MainContractSelection.ipynb"
expected_identity_values = {
    "artifact_role": "method_demo", "producer_notebook": expected_producer,
    "method_source_tag": "demo-dependency",
    "exchange_code": demo_exchange_code, "underlying_code": demo_underlying_code,
    "timezone": ADJUSTMENT_TIMEZONE, "interval_closed": "both",
    "roll_trigger_ratio": demo_roll_trigger_ratio,
}
for identity_key, expected_value in expected_identity_values.items():
    if upstream_demo_identity[identity_key] != expected_value:
        raise RuntimeError(f"上游生成身份 {identity_key} 不匹配，停止依赖计算")
if not (
    date.fromisoformat(upstream_demo_identity["output_start_date"]) <= demo_mapping_read_start_date
    <= demo_adjustment_start_date <= demo_adjustment_end_date <= demo_mapping_read_end_date
    <= date.fromisoformat(upstream_demo_identity["output_end_date"])
):
    raise RuntimeError("读取或复权区间超出上游 demo 生成范围，停止依赖计算")

# 实际内容摘要与上游保存单元格使用完全相同的 Schema、行序和 IPC 口径。
upstream_content_table = upstream_demo_table.take(pa.array(range(upstream_demo_table.num_rows), type=pa.int32())).replace_schema_metadata(MAIN_CONTRACT_DAILY_SCHEMA.metadata)
upstream_content_buffer = pa.BufferOutputStream()
with pa.ipc.new_stream(upstream_content_buffer, upstream_content_table.schema) as upstream_content_writer:
    upstream_content_writer.write_table(upstream_content_table)
actual_upstream_content_sha256 = hashlib.sha256(upstream_content_buffer.getvalue().to_pybytes()).hexdigest()
if actual_upstream_content_sha256 != upstream_demo_identity["output_content_sha256"]:
    raise RuntimeError("上游实际内容摘要不匹配，停止依赖计算")

# 仅读取已明确标记的生成依赖；绘图定义不进入主力映射的数据身份。
upstream_notebook = json.loads(upstream_notebook_path.read_text(encoding="utf-8"))
upstream_dependency_cells = [
    cell for cell in upstream_notebook["cells"]
    if cell["cell_type"] == "code"
    and {"export", "demo-dependency"}.issubset(cell.get("metadata", {}).get("tags", []))
]
if not upstream_dependency_cells:
    raise RuntimeError("上游缺少 demo-dependency 定义，停止依赖计算")
upstream_dependency_source = "\n\n".join("".join(cell["source"]) for cell in upstream_dependency_cells)
actual_upstream_source_sha256 = hashlib.sha256(upstream_dependency_source.encode()).hexdigest()
if actual_upstream_source_sha256 != upstream_demo_identity["method_source_sha256"]:
    raise RuntimeError("上游生成代码摘要不匹配，请重算上游 demo 后再继续")

verified_main_contract_daily_df = upstream_content_table.to_pandas(types_mapper=pd.ArrowDtype)
verified_upstream_identity = upstream_demo_identity
print("生成身份、代码、实际内容和读取范围核对通过；信任上游输出契约。")
print(f"{len(verified_main_contract_daily_df)} 个交易日；{verified_main_contract_daily_df['trading_date'].min()} 至 {verified_main_contract_daily_df['trading_date'].max()}")
display(pd.DataFrame([
    {"field": MAIN_CONTRACT_COLUMN_LABELS[field.name], "type": str(field.type), "nullable": field.nullable}
    for field in MAIN_CONTRACT_DAILY_SCHEMA
]).rename(columns=ADJUSTMENT_COLUMN_LABELS))
display(pd.DataFrame([
    {"table": table_name, **input_identity}
    for table_name, input_identity in verified_upstream_identity["input_identity"].items()
]).rename(columns=ADJUSTMENT_COLUMN_LABELS))
display(pd.DataFrame([{
    "producer": verified_upstream_identity["producer_notebook"],
    "generated_at_utc": verified_upstream_identity["generated_at_utc"],
    "method_version": verified_upstream_identity["method_version"],
    "source_sha256": actual_upstream_source_sha256,
    "content_sha256": actual_upstream_content_sha256,
}]).rename(columns=ADJUSTMENT_COLUMN_LABELS))


生成身份、代码、实际内容和读取范围核对通过；信任上游输出契约。
4067 个交易日；2010-01-04 至 2026-09-30


,field（字段）,type（Arrow 类型）,nullable（是否允许空值）
0,exchange_code（交易所代码）,string,False
1,underlying_code（品种代码）,string,False
2,trading_date（主力生效交易日）,date32[day],False
3,signal_trading_date（决策信号交易日）,date32[day],True
4,incumbent_contract_code（决策前主力合约）,string,True
5,incumbent_delist_date（决策前主力退市日）,date32[day],True
6,challenger_contract_code（挑战者合约）,string,True
7,challenger_delist_date（挑战者退市日）,date32[day],True
8,main_contract_code（当日主力合约）,string,True
9,main_delist_date（当日主力退市日）,date32[day],True


,table（上游表名）,rows（输入行数）,schema_version（契约版本）,sha256（输入内容摘要）
0,dim_futures_bar_calendar,48798,1.4.0,438e13f65c8ed79f64f41900dcabcf18600214e3287bfc...
1,dim_futures_contract_calendar,180687,1.2.0,8ca7a59e22929a6da2da3141de42d9c065215556cb97fd...
2,dim_futures_variety_calendar,4067,1.2.0,bf93bd2c7b3f5135adea88b33767d152dbec77ab7be708...
3,fact_futures_daily,48798,1.5.0,4a9bf0bdfdf7911bc86518305090319fded7116eefe662...


,producer（生成 Notebook）,generated_at_utc（生成时间 UTC）,method_version（方法版本）,source_sha256（算法定义摘要）,content_sha256（输出内容摘要）
0,04_Research/a01_Methods/b01_MainContractSelect...,2026-10-05T21:20:55.616695+00:00,1.0.0,08812d70065523eb7740174686bc74944c2df06aa1025f...,a9b39317ac56ec2505bfbede5b02c5d863742e07e44baa...


### 4.1 固定有效范围与主力片段

根据上游映射动态确定首个已识别交易日，纳入当天分钟；如果初始化前存在未识别前缀，则展示并剔除该前缀。实际计算范围内的未识别日期仍由复权函数拒绝。


In [10]:
mapping_scope_df = verified_main_contract_daily_df.loc[
    verified_main_contract_daily_df["trading_date"].between(demo_mapping_read_start_date, demo_mapping_read_end_date)
].reset_index(drop=True)
identified_dates = mapping_scope_df.loc[mapping_scope_df["main_contract_code"].notna(), "trading_date"]
if identified_dates.empty:
    raise ValueError("请求范围没有已识别主力")
demo_adjustment_start_date = identified_dates.min()
initialization_prefix_df = mapping_scope_df.loc[mapping_scope_df['trading_date'] < demo_adjustment_start_date]
display(initialization_prefix_df.rename(columns=ADJUSTMENT_COLUMN_LABELS))
for scope_key in ("minute_read_trading_dates", "adjustment_trading_dates", "training_trading_dates"):
    demo_scope[scope_key]["start"] = demo_adjustment_start_date.isoformat()
main_segment_daily_df, main_segment_summary_df = build_main_contract_segments(
    mapping_scope_df, exchange_code=demo_exchange_code, underlying_code=demo_underlying_code,
    adjustment_start_date=demo_adjustment_start_date, adjustment_end_date=demo_adjustment_end_date,
)
display(pd.DataFrame(demo_scope).T.rename(columns=ADJUSTMENT_COLUMN_LABELS, index=ADJUSTMENT_COLUMN_LABELS))
display(main_segment_summary_df.rename(columns=ADJUSTMENT_COLUMN_LABELS))


,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,trading_date（bar 归属的交易日，而非其自然日期）,signal_trading_date（决策信号交易日）,incumbent_contract_code（决策前主力合约）,incumbent_delist_date（决策前主力退市日）,challenger_contract_code（挑战者合约）,challenger_delist_date（挑战者退市日）,main_contract_code（当日主力合约）,main_delist_date（当日主力退市日）,incumbent_signal_volume（决策前主力信号日成交量）,challenger_signal_volume（挑战者信号日成交量）,selected_signal_volume（当日主力信号日成交量）,candidate_count（有效正成交量候选数）,unavailable_candidate_count（证据不足候选数）,selection_status（选择状态）,selection_reason（选择原因）,is_roll（是否换约）,roll_trigger_ratio（换约成交量倍数门槛）


,start（起点）,end（终点）
mapping_read（主力映射读取范围）,2010-01-01,2026-09-30
minute_read_trading_dates（分钟读取交易日范围）,2010-01-04,2026-09-30
adjustment_trading_dates（复权交易日范围）,2010-01-04,2026-09-30
training_trading_dates（训练交易日范围）,2010-01-04,2024-12-31
test_trading_dates（测试交易日范围）,2025-01-01,2026-09-30
timezone（时区）,Asia/Shanghai,Asia/Shanghai
interval_closed（区间端点规则）,both,both


,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,segment_id（连续主力片段编号）,main_contract_code（当日主力合约）,segment_start_date（片段首交易日）,segment_end_date（片段末交易日）,trading_day_count（片段交易日数）,is_final_segment（是否最后片段）
0,XSGE,RB,1,RB1005.XSGE,2010-01-04,2010-03-01,36,False
1,XSGE,RB,2,RB1010.XSGE,2010-03-02,2010-07-13,91,False
2,XSGE,RB,3,RB1101.XSGE,2010-07-14,2010-10-11,56,False
3,XSGE,RB,4,RB1105.XSGE,2010-10-12,2011-01-25,75,False
4,XSGE,RB,5,RB1110.XSGE,2011-01-26,2011-07-26,121,False
5,XSGE,RB,6,RB1201.XSGE,2011-07-27,2011-11-02,65,False
6,XSGE,RB,7,RB1205.XSGE,2011-11-03,2012-02-28,77,False
7,XSGE,RB,8,RB1210.XSGE,2012-02-29,2012-07-10,89,False
8,XSGE,RB,9,RB1301.XSGE,2012-07-11,2012-10-17,66,False
9,XSGE,RB,10,RB1305.XSGE,2012-10-18,2013-02-06,77,False


## 6. 日级分段验收与导入

核对日级输出覆盖本次复权区间，片段计数与起止日对应，并验证实际换约日开启新片段。重复合约及未识别日的分段行为在独立小样本检查中验证；真实品种结果是否出现这些情况由下方统计给出。

同名 `.py` 代理加载本页的 `export` 定义。调用方按根模板定位项目，并将 `04_Research` 加入模块路径后，可导入：

```python
from a01_Methods.b02_MainContinuousAdjustment.c01_MainContinuousAdjustment import (
    build_main_contract_segments, prepare_main_contract_minutes,
    adjust_main_contract_minutes, save_main_adjustment_demo,
)
```

导入只加载定义；读取、计算和保存均须显式调用。

In [11]:
expected_adjustment_df = mapping_scope_df.loc[
    mapping_scope_df["trading_date"].between(demo_adjustment_start_date, demo_adjustment_end_date)
]
assert main_segment_daily_df["trading_date"].tolist() == expected_adjustment_df["trading_date"].tolist()
assert len(main_segment_daily_df) == len(expected_adjustment_df)
identified_segments_df = main_segment_daily_df.loc[main_segment_daily_df["segment_id"].notna()]
assert int(main_segment_summary_df["trading_day_count"].sum()) == len(identified_segments_df)
assert main_segment_summary_df["segment_id"].tolist() == list(range(1, len(main_segment_summary_df) + 1))
assert identified_segments_df["trading_date"].between(
    identified_segments_df["segment_start_date"], identified_segments_df["segment_end_date"]
).all()
expected_segment_starts = identified_segments_df.loc[
    identified_segments_df["segment_id"].ne(identified_segments_df["segment_id"].shift()).fillna(True)
]
assert expected_segment_starts["trading_date"].tolist() == main_segment_summary_df["segment_start_date"].tolist()
if main_segment_daily_df["segment_id"].notna().all():
    assert len(main_segment_summary_df) == int(expected_adjustment_df["is_roll"].iloc[1:].fillna(False).sum()) + 1
print("日级覆盖、连续片段编号、起止日与真实换约计数验收通过。")
print("重复担任主力的合约数：", int((main_segment_summary_df.groupby("main_contract_code").size() > 1).sum()))
if not main_segment_summary_df.empty:
    display(main_segment_summary_df.loc[main_segment_summary_df["is_final_segment"]].rename(columns=ADJUSTMENT_COLUMN_LABELS))


日级覆盖、连续片段编号、起止日与真实换约计数验收通过。
重复担任主力的合约数： 0


,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,segment_id（连续主力片段编号）,main_contract_code（当日主力合约）,segment_start_date（片段首交易日）,segment_end_date（片段末交易日）,trading_day_count（片段交易日数）,is_final_segment（是否最后片段）
50,XSGE,RB,51,RB2701.XSGE,2026-09-02,2026-09-30,20,True


### 4.2 逐月读取与全区间主力分钟

正式 silver 使用 PyArrow Dataset、权威 Schema、分区过滤和统一转换入口。读取后信任上游契约，保留本方法需要的正价格、分钟可用性与连接检查。Session 缺失状态直接来自正式日历；两遍读取的月内容摘要必须一致，以保证计算使用同一输入。


In [12]:
import pyarrow.dataset as ds
from config.settings import settings
from config.data_contracts import FUTURES_BAR_CALENDAR_SCHEMA, arrow_to_pandas
from time import perf_counter

minute_dataset = ds.dataset(settings.futures_lake_root / "silver" / FUTURES_MINUTE_SCHEMA.metadata[b"table_name"].decode(),
    format="parquet", schema=FUTURES_MINUTE_SCHEMA,
    partitioning=ds.partitioning(pa.schema([FUTURES_MINUTE_SCHEMA.field(c) for c in FUTURES_MINUTE_SCHEMA.metadata[b"partition_columns"].decode().split(",")]), flavor="hive"))
calendar_dataset = ds.dataset(settings.futures_lake_root / "silver" / FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"table_name"].decode(),
    format="parquet", schema=FUTURES_BAR_CALENDAR_SCHEMA,
    partitioning=ds.partitioning(pa.schema([FUTURES_BAR_CALENDAR_SCHEMA.field(c) for c in FUTURES_BAR_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode().split(",")]), flavor="hive"))
demo_months = pd.period_range(demo_adjustment_start_date, demo_adjustment_end_date, freq="M")
minute_sort_keys = [("bar_at", "ascending"), ("contract_code", "ascending")]
calendar_sort_keys = [("trading_date", "ascending"), ("session_start_at", "ascending"), ("contract_code", "ascending")]
minute_digest, calendar_digest = ArrowContentDigest(FUTURES_MINUTE_SCHEMA), ArrowContentDigest(FUTURES_BAR_CALENDAR_SCHEMA)
main_parts, missing_parts, coverage_parts, unsupported_parts = [], [], [], []
month_inputs, month_calendars = {}, {}
minute_input_rows, calendar_input_rows, anomaly_rows = 0, 0, 0
all_contract_codes = set()
actual_bar_at_start, actual_bar_at_end = None, None
demo_started = perf_counter()
for month_number, period in enumerate(demo_months, 1):
    input_filter = ((ds.field("exchange_code") == demo_exchange_code) & (ds.field("underlying_code") == demo_underlying_code)
        & (ds.field("year") == period.year) & (ds.field("month") == period.month)
        & (ds.field("trading_date") >= demo_adjustment_start_date) & (ds.field("trading_date") <= demo_adjustment_end_date))
    minute_table = minute_dataset.to_table(filter=input_filter, columns=FUTURES_MINUTE_SCHEMA.names).sort_by(minute_sort_keys)
    calendar_table = calendar_dataset.to_table(filter=input_filter & (ds.field("bar_frequency") == "1m"), columns=FUTURES_BAR_CALENDAR_SCHEMA.names).sort_by(calendar_sort_keys)
    month_daily_df = main_segment_daily_df.loc[main_segment_daily_df['trading_date'].between(max(period.start_time.date(), demo_adjustment_start_date), min(period.end_time.date(), demo_adjustment_end_date))]
    if month_daily_df.empty:
        continue
    minute_digest.update(minute_table)
    calendar_digest.update(calendar_table)
    minute_input_rows += minute_table.num_rows
    calendar_input_rows += calendar_table.num_rows
    month_inputs[str(period)] = (input_filter, arrow_content_sha256(minute_table, FUTURES_MINUTE_SCHEMA, minute_sort_keys))
    month_calendars[str(period)] = arrow_to_pandas(calendar_table, FUTURES_BAR_CALENDAR_SCHEMA)
    month_all_df, month_main_df, month_diagnostics = prepare_main_contract_minutes(
        arrow_to_pandas(minute_table, FUTURES_MINUTE_SCHEMA), month_daily_df, month_calendars[str(period)])
    main_parts.append(month_main_df)
    missing_parts.append(month_diagnostics['missing_main_minute_df'])
    coverage_parts.append(month_diagnostics['main_daily_coverage_df'])
    unsupported_parts.append(month_diagnostics['all_contract_times_without_main_df'])
    all_contract_codes.update(month_all_df['contract_code'].unique())
    actual_bar_at_start = month_all_df['bar_at'].min() if actual_bar_at_start is None else actual_bar_at_start
    actual_bar_at_end = month_all_df['bar_at'].max()
    raw = {name: month_all_df[f'raw_{name}'].to_numpy(dtype=float, na_value=np.nan) for name in OHLC_FIELDS}
    anomaly_rows += int(((raw['high'] < np.maximum(raw['open'], raw['close'])) | (raw['low'] > np.minimum(raw['open'], raw['close'])) | (raw['high'] < raw['low'])).sum())
    if month_number == 1 or month_number % 12 == 0 or month_number == len(demo_months):
        print(f"{demo_underlying_code} 读取 {month_number}/{len(demo_months)} 月；{minute_input_rows:,} 行；{perf_counter()-demo_started:.1f} 秒", flush=True)
main_log_minute_df = pd.concat(main_parts, ignore_index=True).sort_values('bar_at').reset_index(drop=True)
missing_main_minute_df = pd.concat(missing_parts, ignore_index=True).sort_values('bar_at').reset_index(drop=True)
main_daily_coverage_df = pd.concat(coverage_parts, ignore_index=True)
unsupported_times_df = pd.concat(unsupported_parts, ignore_index=True)
del main_parts, missing_parts, coverage_parts, unsupported_parts, month_all_df, month_main_df, minute_table, calendar_table
minute_input_identity = {}
for schema, count, digest, sort_keys in ((FUTURES_MINUTE_SCHEMA, minute_input_rows, minute_digest, minute_sort_keys),
                                       (FUTURES_BAR_CALENDAR_SCHEMA, calendar_input_rows, calendar_digest, calendar_sort_keys)):
    minute_input_identity[schema.metadata[b'table_name'].decode()] = {
        'selected_rows': count, 'schema_version': schema.metadata[b'schema_version'].decode(),
        'partition_columns': schema.metadata[b'partition_columns'].decode().split(','),
        'trading_date_start': demo_adjustment_start_date.isoformat(),
        'trading_date_end': demo_adjustment_end_date.isoformat(), 'timezone': ADJUSTMENT_TIMEZONE, 'interval_closed': 'both',
        'content_sha256': digest.hexdigest(), 'schema_sha256': hashlib.sha256(schema.serialize().to_pybytes()).hexdigest(),
        'content_sort_keys': sort_keys,
    }
display(main_daily_coverage_df.loc[main_daily_coverage_df['missing_minute_count'] > 0].rename(columns=ADJUSTMENT_COLUMN_LABELS))


RB 读取 1/201 月；54,000 行；0.4 秒


RB 读取 12/201 月；652,725 行；3.0 秒


RB 读取 24/201 月；1,311,525 行；5.9 秒


RB 读取 36/201 月；1,967,625 行；8.8 秒


RB 读取 48/201 月；2,610,225 行；11.8 秒


...

RB 读取 168/201 月；12,615,030 行；54.3 秒


RB 读取 180/201 月；13,606,830 行；58.3 秒


RB 读取 192/201 月；14,604,210 行；62.4 秒


RB 读取 201/201 月；15,344,910 行；65.4 秒


,trading_date（bar 归属的交易日，而非其自然日期）,main_contract_code（当日主力合约）,segment_id（连续主力片段编号）,expected_minute_count（理论主力分钟数）,observed_minute_count（实际主力分钟数）,missing_minute_count（缺失主力分钟数）
1212,2015-01-05,RB1505.XSGE,16,465,225,240
1244,2015-02-25,RB1505.XSGE,16,465,225,240
1272,2015-04-07,RB1510.XSGE,17,465,225,240
1290,2015-05-04,RB1510.XSGE,17,465,225,240
1325,2015-06-23,RB1510.XSGE,17,465,225,240
1377,2015-09-07,RB1601.XSGE,18,465,225,240
1392,2015-09-28,RB1601.XSGE,18,465,225,240
1395,2015-10-08,RB1601.XSGE,18,465,225,240
1456,2016-01-04,RB1605.XSGE,19,465,225,240
1481,2016-02-15,RB1605.XSGE,19,465,225,240


### 4.3 计算训练／测试偏移

两期按交易日分界分别计算。全部月份共用这组偏移，月界不重置；全合约广播在保存时按月执行。


In [13]:
if main_log_minute_df["bar_at"].duplicated().any():
    raise ValueError("全区间主力分钟时间重叠，不能建立唯一共同偏移")
adjusted_main_minute_df, _, minute_log_offset_df, adjustment_diagnostics = adjust_main_contract_minutes(
    main_log_minute_df, main_log_minute_df, main_segment_daily_df, missing_main_minute_df,
    test_start_date=demo_test_start_date)
roll_diagnostics_df = adjustment_diagnostics['roll_diagnostics_df']
display(adjustment_diagnostics['segment_diagnostics_df'].rename(columns=ADJUSTMENT_COLUMN_LABELS))
display(roll_diagnostics_df.rename(columns=ADJUSTMENT_COLUMN_LABELS))
print('实际主力分钟：', len(main_log_minute_df), '；缺失主力分钟：', len(missing_main_minute_df))


,segment_id（连续主力片段编号）,main_contract_code（当日主力合约）,first_bar_at（片段首实际分钟时点）,last_bar_at（片段末实际分钟时点）,first_trading_date（片段首实际分钟所属交易日）,last_trading_date（片段末实际分钟所属交易日）,minute_count（实际主力分钟数）,backward_log_shift（训练累计后续换约对数偏移）,forward_log_shift（测试累计过去换约对数偏移）,adjustment_period（复权所属训练或测试期）,missing_minute_count（缺失主力分钟数）,is_final_segment（是否最后片段）
0,1,RB1005.XSGE,2010-01-04 09:01:00+08:00,2010-03-01 15:00:00+08:00,2010-01-04,2010-03-01,8100,-0.307746,0.000000,train,0,False
1,2,RB1010.XSGE,2010-03-02 09:01:00+08:00,2010-07-13 15:00:00+08:00,2010-03-02,2010-07-13,20475,-0.410823,0.000000,train,0,False
2,3,RB1101.XSGE,2010-07-14 09:01:00+08:00,2010-10-11 15:00:00+08:00,2010-07-14,2010-10-11,12600,-0.456484,0.000000,train,0,False
3,4,RB1105.XSGE,2010-10-12 09:01:00+08:00,2011-01-25 15:00:00+08:00,2010-10-12,2011-01-25,16875,-0.489118,0.000000,train,0,False
4,5,RB1110.XSGE,2011-01-26 09:01:00+08:00,2011-07-26 15:00:00+08:00,2011-01-26,2011-07-26,27225,-0.507668,0.000000,train,0,False
5,6,RB1201.XSGE,2011-07-27 09:01:00+08:00,2011-11-02 15:00:00+08:00,2011-07-27,2011-11-02,14625,-0.506442,0.000000,train,0,False
6,7,RB1205.XSGE,2011-11-03 09:01:00+08:00,2012-02-28 15:00:00+08:00,2011-11-03,2012-02-28,17325,-0.49531,0.000000,train,0,False
7,8,RB1210.XSGE,2012-02-29 09:01:00+08:00,2012-07-10 15:00:00+08:00,2012-02-29,2012-07-10,20025,-0.493213,0.000000,train,0,False
8,9,RB1301.XSGE,2012-07-11 09:01:00+08:00,2012-10-17 15:00:00+08:00,2012-07-11,2012-10-17,14850,-0.476554,0.000000,train,0,False
9,10,RB1305.XSGE,2012-10-18 09:01:00+08:00,2013-02-06 15:00:00+08:00,2012-10-18,2013-02-06,17325,-0.485312,0.000000,train,0,False


,adjustment_period（复权所属训练或测试期）,old_segment_id（换约前片段编号）,new_segment_id（换约后片段编号）,old_contract_code（换约前合约）,new_contract_code（换约后合约）,old_bar_at（换约前末分钟时点）,new_bar_at（换约后首分钟时点）,gap_log（新合约首开盘减旧合约末收盘的对数价差）,old_linear_log_deduction（换约前末分钟线性扣除量）,new_linear_log_deduction（换约后首分钟线性扣除量）,expected_residual_log（换约边界预期残余对数价差）,actual_residual_log（换约边界实际残余对数价差）
0,train,1,2,RB1005.XSGE,RB1010.XSGE,2010-03-01 15:00:00+08:00,2010-03-02 09:01:00+08:00,0.103077,-0.305548,-0.305548,-2.713631e-07,-2.713631e-07
1,train,2,3,RB1010.XSGE,RB1101.XSGE,2010-07-13 15:00:00+08:00,2010-07-14 09:01:00+08:00,0.045661,-0.299992,-0.299992,-2.713631e-07,-2.713631e-07
2,train,3,4,RB1101.XSGE,RB1105.XSGE,2010-10-11 15:00:00+08:00,2010-10-12 09:01:00+08:00,0.032634,-0.296573,-0.296572,-2.713631e-07,-2.713631e-07
3,train,4,5,RB1105.XSGE,RB1110.XSGE,2011-01-25 15:00:00+08:00,2011-01-26 09:01:00+08:00,0.018550,-0.291993,-0.291993,-2.713631e-07,-2.713631e-07
4,train,5,6,RB1110.XSGE,RB1201.XSGE,2011-07-26 15:00:00+08:00,2011-07-27 09:01:00+08:00,-0.001226,-0.284606,-0.284605,-2.713631e-07,-2.713631e-07
5,train,6,7,RB1201.XSGE,RB1205.XSGE,2011-11-02 15:00:00+08:00,2011-11-03 09:01:00+08:00,-0.011132,-0.280637,-0.280637,-2.713631e-07,-2.713631e-07
6,train,7,8,RB1205.XSGE,RB1210.XSGE,2012-02-28 15:00:00+08:00,2012-02-29 09:01:00+08:00,-0.002098,-0.275936,-0.275935,-2.713631e-07,-2.713631e-07
7,train,8,9,RB1210.XSGE,RB1301.XSGE,2012-07-10 15:00:00+08:00,2012-07-11 09:01:00+08:00,-0.016659,-0.270502,-0.270501,-2.713631e-07,-2.713631e-07
8,train,9,10,RB1301.XSGE,RB1305.XSGE,2012-10-17 15:00:00+08:00,2012-10-18 09:01:00+08:00,0.008758,-0.266472,-0.266472,-2.713631e-07,-2.713631e-07
9,train,10,11,RB1305.XSGE,RB1310.XSGE,2013-02-06 15:00:00+08:00,2013-02-07 09:01:00+08:00,0.021905,-0.261770,-0.261770,-2.713631e-07,-2.713631e-07


实际主力分钟： 1278915 ；缺失主力分钟： 18360


## 5. 确定性小例子与边界行为

三天主力为 A → B → A，每天两分钟，验证重复合约仍分成三个片段。另检查片段首末理论分钟缺失时使用实际锚点、单片段、单点和两点边界、缺失 log、同一分钟有两个非主力合约的无偏移情况、零价格的原值保留与派生字段置空、负价格拒绝及锚点不可用。测试行情独立构造，不写入正式湖。

In [14]:
def make_small_adjustment_example(contract_sequence, minutes_per_day):
    """构造少量固定分钟，用于核对复权公式和输入边界。"""
    trading_dates = [date(2020, 1, 2), date(2020, 1, 3), date(2020, 1, 6)][:len(contract_sequence)]
    codes = ["RB2001.XSGE", "RB2005.XSGE"]
    daily_df = pd.DataFrame({
        "exchange_code": "XSGE", "underlying_code": "RB", "trading_date": trading_dates,
        "signal_trading_date": trading_dates, "main_contract_code": contract_sequence,
        "selection_status": "example", "is_roll": [False] + [True] * (len(trading_dates) - 1),
    })
    daily_df, summary_df = build_main_contract_segments(daily_df, exchange_code="XSGE", underlying_code="RB")
    rows, sessions = [], []
    first_log_open_by_day = [2.0, 3.0, 2.5]
    for day_index, trading_date in enumerate(trading_dates):
        session_start_at = pd.Timestamp(trading_date, tz=ADJUSTMENT_TIMEZONE) + pd.Timedelta(hours=9)
        for contract_index, contract_code in enumerate(codes):
            sessions.append({
                "bar_frequency": "1m", "contract_code": contract_code, "exchange_code": "XSGE",
                "underlying_code": "RB", "trading_date": trading_date, "session_number": 1,
                "session_start_at": session_start_at, "session_end_at": session_start_at + pd.Timedelta(minutes=minutes_per_day),
                "schedule_status": "example", "schedule_signal_reason": "synthetic",
                "is_fetch_required": True, "is_fetch_completed": True,
                "expected_bar_count": minutes_per_day, "actual_bar_count": minutes_per_day,
                "missing_bar_count": 0, "quality_status": "example", "quality_reason": "synthetic",
            })
            for minute_index in range(minutes_per_day):
                base_log = first_log_open_by_day[day_index] + minute_index * 0.1
                if contract_code != contract_sequence[day_index]:
                    base_log += 0.2
                rows.append({
                    "contract_code": contract_code, "exchange_code": "XSGE", "underlying_code": "RB",
                    "trading_date": trading_date, "session_number": 1,
                    "bar_at": session_start_at + pd.Timedelta(minutes=minute_index + 1),
                    "open": np.exp(base_log), "high": np.exp(base_log + .05),
                    "low": np.exp(base_log - .04), "close": np.exp(base_log + .02),
                    "volume": 100.0 + contract_index, "money": 10000.0 + minute_index,
                    "open_interest": 1000.0 + contract_index,
                    "source": "synthetic", "updated_at": pd.Timestamp("2020-01-10", tz="UTC"),
                    "year": 2020, "month": 1,
                })
    minute_df = pa.Table.from_pylist(rows, schema=FUTURES_MINUTE_SCHEMA).to_pandas(types_mapper=pd.ArrowDtype)
    return minute_df, daily_df, pd.DataFrame(sessions)

small_minute_df, small_daily_df, small_calendar_df = make_small_adjustment_example(
    ["RB2001.XSGE", "RB2005.XSGE", "RB2001.XSGE"], 2,
)
small_all_df, small_main_df, small_preparation = prepare_main_contract_minutes(small_minute_df, small_daily_df, small_calendar_df)
small_adjusted_main_df, small_adjusted_all_df, small_offset_df, small_adjustment = adjust_main_contract_minutes(
    small_main_df, small_all_df, small_daily_df, small_preparation["missing_main_minute_df"],
 test_start_date=date(2025, 1, 1))
assert small_daily_df["segment_id"].tolist() == [1, 2, 3]
np.testing.assert_allclose(small_adjustment["roll_diagnostics_df"]["gap_log"], [.88, -.62], atol=1e-14)
np.testing.assert_allclose(small_offset_df["backward_log_shift"].to_numpy(dtype=float), [.26, .26, -.62, -.62, 0, 0], atol=1e-14)
np.testing.assert_allclose(small_offset_df["linear_log_deduction"].to_numpy(dtype=float), [.26, .208, .156, .104, .052, 0], atol=1e-14)
np.testing.assert_allclose(small_offset_df["log_adjustment"].to_numpy(dtype=float), [0, .052, -.776, -.724, -.052, 0], atol=1e-14)
# 每个相邻间隔都增加 .052；换约处先扣除原 gap，首尾 OHLC 精确保留。
small_roll_mask = small_main_df["segment_id"].to_numpy()[1:] != small_main_df["segment_id"].to_numpy()[:-1]
small_raw_steps = small_main_df["log_open"].to_numpy(dtype=float)[1:] - small_main_df["log_close"].to_numpy(dtype=float)[:-1]
small_adjusted_steps = small_adjusted_main_df["adjusted_log_open"].to_numpy(dtype=float)[1:] - small_adjusted_main_df["adjusted_log_close"].to_numpy(dtype=float)[:-1]
np.testing.assert_allclose(small_adjusted_steps, np.where(small_roll_mask, 0, small_raw_steps) + .052, atol=1e-14)
for name in OHLC_FIELDS:
    assert small_adjusted_main_df[f"adjusted_{name}"].iloc[[0, -1]].equals(small_main_df[f"raw_{name}"].iloc[[0, -1]])
display(small_offset_df.rename(columns=ADJUSTMENT_COLUMN_LABELS))
display(small_adjustment["roll_diagnostics_df"].rename(columns=ADJUSTMENT_COLUMN_LABELS))

for contracts, minutes, expected_intervals in ((["RB2001.XSGE"], 1, 0), (["RB2001.XSGE"], 3, 2), (["RB2001.XSGE", "RB2005.XSGE"], 1, 1)):
    test_minute_df, test_daily_df, test_calendar_df = make_small_adjustment_example(contracts, minutes)
    test_all_df, test_main_df, test_preparation = prepare_main_contract_minutes(test_minute_df, test_daily_df, test_calendar_df)
    test_adjusted_main_df, test_adjusted_all_df, test_offset_df, test_adjustment = adjust_main_contract_minutes(
        test_main_df, test_all_df, test_daily_df, test_preparation["missing_main_minute_df"],
     test_start_date=date(2025, 1, 1))
    assert test_adjustment["interval_count"] == expected_intervals
    assert test_offset_df["log_adjustment"].eq(0).all()
    if expected_intervals == 1:
        np.testing.assert_allclose(test_adjustment["roll_diagnostics_df"]["actual_residual_log"], [test_adjustment["total_log_gap"]], atol=1e-14)
        print("两点、一个间隔的分摊量：", test_adjustment["per_interval_log_gap"])

# 同一时点只有非主力合约行情：原价保留，调整结果为空，不借用相邻偏移。
test_minute_df, test_daily_df, test_calendar_df = make_small_adjustment_example(["RB2001.XSGE", "RB2005.XSGE"], 3)
interior_bar_at = test_minute_df["bar_at"].min() + pd.Timedelta(minutes=1)
partial_minute_df = test_minute_df.loc[~(test_minute_df["contract_code"].eq("RB2001.XSGE") & test_minute_df["bar_at"].eq(interior_bar_at))].copy()
# 非锚点 OHLC 空值保持为空。
partial_minute_df.loc[partial_minute_df["bar_at"].eq(interior_bar_at), "high"] = pd.NA
# 在同一无主力时点增加第二个非主力合约，原始 log 价差应有限。
extra_non_main_df = partial_minute_df.loc[partial_minute_df["bar_at"].eq(interior_bar_at)].copy()
extra_non_main_df["contract_code"] = pd.array(["RB2010.XSGE"], dtype=pd.ArrowDtype(pa.string()))
for name in OHLC_FIELDS:
    extra_non_main_df[name] = extra_non_main_df[name] * np.exp(.3)
partial_minute_df = pd.concat([partial_minute_df, extra_non_main_df], ignore_index=True)
partial_all_df, partial_main_df, partial_preparation = prepare_main_contract_minutes(partial_minute_df, test_daily_df, test_calendar_df)
partial_adjusted_main_df, partial_adjusted_all_df, partial_offset_df, partial_adjustment = adjust_main_contract_minutes(
    partial_main_df, partial_all_df, test_daily_df, partial_preparation["missing_main_minute_df"],
 test_start_date=date(2025, 1, 1))
unsupported_rows_df = partial_adjusted_all_df.loc[partial_adjusted_all_df["bar_at"].eq(interior_bar_at)]
assert len(partial_preparation["missing_main_minute_df"]) == 1
assert len(unsupported_rows_df) == 2
assert unsupported_rows_df["log_close"].notna().all()
assert unsupported_rows_df["adjustment_status"].eq("no_main_offset").all()
assert unsupported_rows_df[[f"adjusted_{name}" for name in OHLC_FIELDS]].isna().all().all()
assert unsupported_rows_df[["backward_log_shift", "forward_log_shift", "linear_log_deduction", "log_adjustment"]].isna().all().all()
assert partial_adjusted_all_df[ALL_CONTRACT_LOG_MINUTE_SCHEMA.names].equals(partial_all_df)
assert len(partial_adjusted_all_df) == len(partial_all_df)
partial_same_time = partial_adjusted_all_df["bar_at"].eq(partial_adjusted_all_df["bar_at"].shift()).fillna(False).to_numpy(dtype=bool)[1:]
partial_shift = partial_adjusted_all_df["log_adjustment"].to_numpy(dtype=float, na_value=np.nan)
for name in OHLC_FIELDS:
    original = partial_adjusted_all_df[f"log_{name}"].to_numpy(dtype=float, na_value=np.nan)
    adjusted = partial_adjusted_all_df[f"adjusted_log_{name}"].to_numpy(dtype=float, na_value=np.nan)
    valid_pairs = (partial_same_time & np.isfinite(partial_shift[:-1]) & np.isfinite(partial_shift[1:])
                   & np.isfinite(original[:-1]) & np.isfinite(original[1:])
                   & np.isfinite(adjusted[:-1]) & np.isfinite(adjusted[1:]))
    np.testing.assert_allclose(np.diff(adjusted)[valid_pairs], np.diff(original)[valid_pairs], atol=2e-14, rtol=1e-10)
assert unsupported_rows_df[[f"adjusted_log_{name}" for name in OHLC_FIELDS]].isna().all().all()
assert unsupported_rows_df["log_high"].isna().all()
assert len(partial_adjustment["all_contract_times_without_offset_df"]) == 1
for quantity in QUANTITY_FIELDS:
    assert partial_adjusted_all_df[quantity].equals(partial_all_df[quantity])

# 零 low 覆盖主力/非主力、非零偏移及首尾零偏移，原值与数量均保留。
zero_minute_df = small_minute_df.copy()
zero_minute_df["low"] = pd.array(np.zeros(len(zero_minute_df)), dtype=small_minute_df["low"].dtype)
zero_all_df, zero_main_df, zero_preparation = prepare_main_contract_minutes(zero_minute_df, small_daily_df, small_calendar_df)
_, zero_adjusted_all_df, _, _ = adjust_main_contract_minutes(zero_main_df, zero_all_df, small_daily_df, zero_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
assert zero_adjusted_all_df["raw_low"].eq(0).all()
assert zero_adjusted_all_df[["log_low", "adjusted_log_low", "adjusted_low"]].isna().all().all()
assert zero_adjusted_all_df["log_adjustment"].eq(0).any()
assert zero_adjusted_all_df["log_adjustment"].ne(0).any()
for quantity in QUANTITY_FIELDS:
    assert zero_adjusted_all_df[quantity].equals(small_adjusted_all_df[quantity])
# 首行 open 不参与换约缺口；零值仍不得被零偏移恢复成复权价格 0。
zero_open_minute_df = small_minute_df.copy()
zero_open_minute_df.loc[0, "open"] = 0.0
zero_open_all_df, zero_open_main_df, zero_open_preparation = prepare_main_contract_minutes(zero_open_minute_df, small_daily_df, small_calendar_df)
_, zero_open_adjusted_df, _, _ = adjust_main_contract_minutes(zero_open_main_df, zero_open_all_df, small_daily_df, zero_open_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
zero_open_row_df = zero_open_adjusted_df.loc[zero_open_adjusted_df["raw_open"].eq(0)]
assert zero_open_row_df[["log_open", "adjusted_log_open", "adjusted_open"]].isna().all().all()
assert zero_open_row_df["log_adjustment"].eq(0).all()

bad_minute_df = small_minute_df.copy()
bad_minute_df.loc[0, "open"] = -1.0
try:
    prepare_main_contract_minutes(bad_minute_df, small_daily_df, small_calendar_df)
except ValueError as error:
    assert "负价格" in str(error)
else:
    raise AssertionError("负价格未被拒绝")

# 每片段有两条主力分钟：逐项去掉首条或末条，保留实际锚点并记录缺口。
for missing_edge in small_main_df[["contract_code", "bar_at"]].itertuples(index=False):
    edge_minute_df = small_minute_df.loc[~(small_minute_df["contract_code"].eq(missing_edge.contract_code) & small_minute_df["bar_at"].eq(missing_edge.bar_at))]
    edge_all_df, edge_main_df, edge_preparation = prepare_main_contract_minutes(edge_minute_df, small_daily_df, small_calendar_df)
    edge_adjusted_main_df, edge_adjusted_all_df, edge_offset_df, edge_diagnostics = adjust_main_contract_minutes(
        edge_main_df, edge_all_df, small_daily_df, edge_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
    assert len(edge_preparation["missing_main_minute_df"]) == 1
    assert edge_diagnostics["missing_main_minute_df"].equals(edge_preparation["missing_main_minute_df"])
    assert len(edge_offset_df) == len(small_main_df) - 1
    assert len(edge_adjusted_all_df) == len(edge_minute_df)
    assert edge_offset_df["log_adjustment"].iloc[0] == edge_offset_df["log_adjustment"].iloc[-1] == 0
    actual_bounds_df = edge_main_df.groupby("segment_id", sort=False)["bar_at"].agg(first_bar_at="min", last_bar_at="max").reset_index()
    assert edge_diagnostics["segment_diagnostics_df"][["segment_id", "first_bar_at", "last_bar_at"]].equals(actual_bounds_df)
    # 同时点非主力行情保留，但不借用相邻分钟偏移。
    assert edge_adjusted_all_df.loc[edge_adjusted_all_df["bar_at"].eq(missing_edge.bar_at), "log_adjustment"].isna().all()
new_anchor_at = small_main_df.loc[small_main_df["segment_id"].eq(2), "bar_at"].min()
for unavailable_anchor in (pd.NA, 0.0):
    anchor_minute_df = small_minute_df.copy()
    anchor_minute_df.loc[anchor_minute_df["bar_at"].eq(new_anchor_at) & anchor_minute_df["contract_code"].eq("RB2005.XSGE"), "open"] = unavailable_anchor
    anchor_all_df, anchor_main_df, anchor_preparation = prepare_main_contract_minutes(anchor_minute_df, small_daily_df, small_calendar_df)
    try:
        adjust_main_contract_minutes(anchor_main_df, anchor_all_df, small_daily_df, anchor_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
    except ValueError as error:
        assert "锚点缺失" in str(error)
    else:
        raise AssertionError("换约空值或零价格锚点未被拒绝")
print("确定性公式、重复合约分段、单点/两点、无偏移时点、空值、零值派生字段、负价格及实际边界锚点检查通过。")

# 原价空值在具备偏移的时间点也保持为空；主力数量取各日选中合约。
assert small_main_df["volume"].tolist() == [100.0, 100.0, 101.0, 101.0, 100.0, 100.0]
null_minute_df = small_minute_df.copy()
null_minute_df.loc[0, "high"] = pd.NA
null_all_df, null_main_df, null_preparation = prepare_main_contract_minutes(null_minute_df, small_daily_df, small_calendar_df)
_, null_adjusted_all_df, _, _ = adjust_main_contract_minutes(null_main_df, null_all_df, small_daily_df, null_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
assert null_adjusted_all_df["raw_high"].isna().sum() == 1
assert null_adjusted_all_df["log_high"].isna().sum() == 1
assert null_adjusted_all_df["adjusted_log_high"].isna().sum() == 1
assert null_adjusted_all_df["adjusted_high"].isna().sum() == 1

# 两个不同主力合约拥有相同时间戳：即使合约主键不同也拒绝公共网格重叠。
overlap_minute_df, overlap_daily_df, overlap_calendar_df = make_small_adjustment_example(["RB2001.XSGE", "RB2005.XSGE"], 2)
overlap_minute_df = overlap_minute_df.merge(overlap_daily_df[["trading_date", "main_contract_code"]], on="trading_date", validate="many_to_one")
overlap_minute_df = overlap_minute_df.loc[overlap_minute_df["contract_code"].eq(overlap_minute_df["main_contract_code"])].drop(columns="main_contract_code")
second_day_mask = overlap_minute_df["trading_date"].eq(date(2020, 1, 3))
overlap_minute_df.loc[second_day_mask, "bar_at"] = overlap_minute_df.loc[second_day_mask, "bar_at"] - pd.Timedelta(days=1)
try:
    prepare_main_contract_minutes(overlap_minute_df, overlap_daily_df, overlap_calendar_df)
except ValueError as error:
    assert "时间重叠" in str(error)
else:
    raise AssertionError("主力时间重叠未被拒绝")

missing_segment_minute_df = test_minute_df.loc[~(test_minute_df["trading_date"].eq(date(2020, 1, 3)) & test_minute_df["contract_code"].eq("RB2005.XSGE"))]
missing_segment_all_df, missing_segment_main_df, missing_segment_preparation = prepare_main_contract_minutes(missing_segment_minute_df, test_daily_df, test_calendar_df)
try:
    adjust_main_contract_minutes(missing_segment_main_df, missing_segment_all_df, test_daily_df, missing_segment_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
except ValueError as error:
    assert "整段缺行情" in str(error)
else:
    raise AssertionError("整段缺行情未被拒绝")
unresolved_daily_df = test_daily_df.copy()
unresolved_daily_df.loc[1, ["main_contract_code", "segment_id"]] = pd.NA
unresolved_all_df, unresolved_main_df, unresolved_preparation = prepare_main_contract_minutes(test_minute_df, unresolved_daily_df, test_calendar_df)
assert len(unresolved_preparation["unresolved_main_daily_df"]) == 1
try:
    adjust_main_contract_minutes(unresolved_main_df, unresolved_all_df, unresolved_daily_df, unresolved_preparation["missing_main_minute_df"], test_start_date=date(2025, 1, 1))
except ValueError as error:
    assert "每日主力全部识别" in str(error)
else:
    raise AssertionError("未识别交易日未被拒绝")
print("补充检查通过：有偏移处 null 保留、主力数量身份、时间重叠、整段缺行情和未识别日。")


,trading_date（bar 归属的交易日，而非其自然日期）,bar_at（一分钟 bar 的结束时刻）,main_contract_code（当日主力合约）,segment_id（连续主力片段编号）,backward_log_shift（训练累计后续换约对数偏移）,forward_log_shift（测试累计过去换约对数偏移）,linear_log_deduction（线性扣除的对数偏移）,log_adjustment（共同复权对数偏移）
0,2020-01-02,2020-01-02 09:01:00+08:00,RB2001.XSGE,1,0.26,0.0,0.26,0.0
1,2020-01-02,2020-01-02 09:02:00+08:00,RB2001.XSGE,1,0.26,0.0,0.208,0.052
2,2020-01-03,2020-01-03 09:01:00+08:00,RB2005.XSGE,2,-0.62,0.0,0.156,-0.776
3,2020-01-03,2020-01-03 09:02:00+08:00,RB2005.XSGE,2,-0.62,0.0,0.104,-0.724
4,2020-01-06,2020-01-06 09:01:00+08:00,RB2001.XSGE,3,0.0,0.0,0.052,-0.052
5,2020-01-06,2020-01-06 09:02:00+08:00,RB2001.XSGE,3,0.0,0.0,0.0,0.0


,adjustment_period（复权所属训练或测试期）,old_segment_id（换约前片段编号）,new_segment_id（换约后片段编号）,old_contract_code（换约前合约）,new_contract_code（换约后合约）,old_bar_at（换约前末分钟时点）,new_bar_at（换约后首分钟时点）,gap_log（新合约首开盘减旧合约末收盘的对数价差）,old_linear_log_deduction（换约前末分钟线性扣除量）,new_linear_log_deduction（换约后首分钟线性扣除量）,expected_residual_log（换约边界预期残余对数价差）,actual_residual_log（换约边界实际残余对数价差）
0,train,1,2,RB2001.XSGE,RB2005.XSGE,2020-01-02 09:02:00+08:00,2020-01-03 09:01:00+08:00,0.88,0.208,0.156,0.052,0.052
1,train,2,3,RB2005.XSGE,RB2001.XSGE,2020-01-03 09:02:00+08:00,2020-01-06 09:01:00+08:00,-0.62,0.104,0.052,0.052,0.052


两点、一个间隔的分摊量： 0.98


确定性公式、重复合约分段、单点/两点、无偏移时点、空值、零值派生字段、负价格及实际边界锚点检查通过。
补充检查通过：有偏移处 null 保留、主力数量身份、时间重叠、整段缺行情和未识别日。


## 6. 训练／测试数学核对

训练首尾与测试首个实际分钟保持原价；训练线性回扣只按训练分钟，测试回扣为零；期内换约残差分别为训练分摊量与零。写入时验证全合约共同偏移、原价及数量保留。


In [15]:
train_offsets = minute_log_offset_df.loc[minute_log_offset_df['trading_date'] < demo_test_start_date]
test_offsets = minute_log_offset_df.loc[minute_log_offset_df['trading_date'] >= demo_test_start_date]
anchor_times = []
if not train_offsets.empty:
    assert train_offsets['log_adjustment'].iloc[0] == train_offsets['log_adjustment'].iloc[-1] == 0
    assert train_offsets['forward_log_shift'].eq(0).all()
    np.testing.assert_allclose(np.diff(train_offsets['linear_log_deduction'].to_numpy(dtype=float)),
        -adjustment_diagnostics['per_interval_log_gap'], atol=2e-14, rtol=1e-10)
    anchor_times.extend(train_offsets['bar_at'].iloc[[0, -1]])
if not test_offsets.empty:
    assert test_offsets['log_adjustment'].iloc[0] == 0
    assert test_offsets['backward_log_shift'].eq(0).all()
    assert test_offsets['linear_log_deduction'].eq(0).all()
    anchor_times.append(test_offsets['bar_at'].iloc[0])
anchors_df = adjusted_main_minute_df.loc[adjusted_main_minute_df['bar_at'].isin(anchor_times)]
for name in OHLC_FIELDS:
    positive = anchors_df[f'raw_{name}'].gt(0).fillna(False)
    assert anchors_df.loc[positive, f'adjusted_{name}'].equals(anchors_df.loc[positive, f'raw_{name}'])
np.testing.assert_allclose(roll_diagnostics_df['actual_residual_log'], roll_diagnostics_df['expected_residual_log'], atol=2e-14, rtol=1e-10)
display(adjusted_main_minute_df.head(3).rename(columns=ADJUSTMENT_COLUMN_LABELS))
display(adjusted_main_minute_df.tail(3).rename(columns=ADJUSTMENT_COLUMN_LABELS))
print('训练首尾、测试起点、两期回扣及期内换约衔接验收通过。')


,contract_code（项目标准固定月份合约代码）,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,trading_date（bar 归属的交易日，而非其自然日期）,session_number（bar 所属 Session 编号）,bar_at（一分钟 bar 的结束时刻）,raw_open（原始本分钟开盘价）,raw_high（原始本分钟最高价）,raw_low（原始本分钟最低价）,raw_close（原始本分钟收盘价）,volume（本分钟成交量）,money（本分钟成交金额）,open_interest（本分钟结束时持仓量）,source（行情来源与关键参数）,updated_at（本行最后写入时间）,year（归属交易年份）,month（归属交易月份）,log_open（原始本分钟开盘价自然对数）,log_high（原始本分钟最高价自然对数）,log_low（原始本分钟最低价自然对数）,log_close（原始本分钟收盘价自然对数）,main_contract_code（当日主力合约）,segment_id（连续主力片段编号）,is_main（是否当日主力）,backward_log_shift（训练累计后续换约对数偏移）,forward_log_shift（测试累计过去换约对数偏移）,linear_log_deduction（线性扣除的对数偏移）,log_adjustment（共同复权对数偏移）,adjustment_status（复权状态）,adjusted_log_open（复权本分钟开盘价自然对数）,adjusted_log_high（复权本分钟最高价自然对数）,adjusted_log_low（复权本分钟最低价自然对数）,adjusted_log_close（复权本分钟收盘价自然对数）,adjusted_open（复权本分钟开盘价）,adjusted_high（复权本分钟最高价）,adjusted_low（复权本分钟最低价）,adjusted_close（复权本分钟收盘价）
0,RB1005.XSGE,XSGE,RB,2010-01-04,1,2010-01-04 09:01:00+08:00,4523.0,4540.0,4523.0,4535.0,21988.0,996915100.0,914162.0,JQData_get_price_1m_skip_paused,2026-08-02 13:53:23+00:00,2010,1,8.416931,8.420682,8.416931,8.41958,RB1005.XSGE,1,True,-0.307746,0.0,-0.307746,0.0,adjusted,8.416931,8.420682,8.416931,8.41958,4523.0,4540.0,4523.0,4535.0
1,RB1005.XSGE,XSGE,RB,2010-01-04,1,2010-01-04 09:02:00+08:00,4535.0,4535.0,4524.0,4528.0,16676.0,755024100.0,913964.0,JQData_get_price_1m_skip_paused,2026-08-02 13:53:23+00:00,2010,1,8.41958,8.41958,8.417152,8.418036,RB1005.XSGE,1,True,-0.307746,0.0,-0.307746,-0.0,adjusted,8.41958,8.41958,8.417152,8.418035,4534.998769,4534.998769,4523.998772,4527.998771
2,RB1005.XSGE,XSGE,RB,2010-01-04,1,2010-01-04 09:03:00+08:00,4528.0,4540.0,4528.0,4538.0,15176.0,688346280.0,914710.0,JQData_get_price_1m_skip_paused,2026-08-02 13:53:23+00:00,2010,1,8.418036,8.420682,8.418036,8.420242,RB1005.XSGE,1,True,-0.307746,0.0,-0.307745,-0.000001,adjusted,8.418035,8.420682,8.418035,8.420241,4527.997543,4539.997536,4527.997543,4537.997537


,contract_code（项目标准固定月份合约代码）,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,trading_date（bar 归属的交易日，而非其自然日期）,session_number（bar 所属 Session 编号）,bar_at（一分钟 bar 的结束时刻）,raw_open（原始本分钟开盘价）,raw_high（原始本分钟最高价）,raw_low（原始本分钟最低价）,raw_close（原始本分钟收盘价）,volume（本分钟成交量）,money（本分钟成交金额）,open_interest（本分钟结束时持仓量）,source（行情来源与关键参数）,updated_at（本行最后写入时间）,year（归属交易年份）,month（归属交易月份）,log_open（原始本分钟开盘价自然对数）,log_high（原始本分钟最高价自然对数）,log_low（原始本分钟最低价自然对数）,log_close（原始本分钟收盘价自然对数）,main_contract_code（当日主力合约）,segment_id（连续主力片段编号）,is_main（是否当日主力）,backward_log_shift（训练累计后续换约对数偏移）,forward_log_shift（测试累计过去换约对数偏移）,linear_log_deduction（线性扣除的对数偏移）,log_adjustment（共同复权对数偏移）,adjustment_status（复权状态）,adjusted_log_open（复权本分钟开盘价自然对数）,adjusted_log_high（复权本分钟最高价自然对数）,adjusted_log_low（复权本分钟最低价自然对数）,adjusted_log_close（复权本分钟收盘价自然对数）,adjusted_open（复权本分钟开盘价）,adjusted_high（复权本分钟最高价）,adjusted_low（复权本分钟最低价）,adjusted_close（复权本分钟收盘价）
1278912,RB2701.XSGE,XSGE,RB,2026-09-30,4,2026-09-30 14:58:00+08:00,3112.0,3113.0,3112.0,3112.0,1974.0,61439540.0,1595528.0,JQData_get_price_1m_skip_paused_fq_none,2026-10-02 08:04:14.866421+00:00,2026,9,8.043021,8.043342,8.043021,8.043021,RB2701.XSGE,51,True,0.0,-0.078652,0.0,-0.078652,adjusted,7.964369,7.96469,7.964369,7.964369,2876.612728,2877.53709,2876.612728,2876.612728
1278913,RB2701.XSGE,XSGE,RB,2026-09-30,4,2026-09-30 14:59:00+08:00,3113.0,3113.0,3112.0,3113.0,3899.0,121364160.0,1594314.0,JQData_get_price_1m_skip_paused_fq_none,2026-10-02 08:04:14.866421+00:00,2026,9,8.043342,8.043342,8.043021,8.043342,RB2701.XSGE,51,True,0.0,-0.078652,0.0,-0.078652,adjusted,7.96469,7.96469,7.964369,7.96469,2877.53709,2877.53709,2876.612728,2877.53709
1278914,RB2701.XSGE,XSGE,RB,2026-09-30,4,2026-09-30 15:00:00+08:00,3113.0,3113.0,3112.0,3112.0,5452.0,169692770.0,1592266.0,JQData_get_price_1m_skip_paused_fq_none,2026-10-02 08:04:14.866421+00:00,2026,9,8.043342,8.043342,8.043021,8.043021,RB2701.XSGE,51,True,0.0,-0.078652,0.0,-0.078652,adjusted,7.96469,7.96469,7.964369,7.964369,2877.53709,2877.53709,2876.612728,2876.612728


训练首尾、测试起点、两期回扣及期内换约衔接验收通过。


## 7. 逐月广播、保存与复读

每个品种在本方法 `c01_MainContinuousAdjustment_DemoData/` 中保存一份完整全合约长表：`c01_MainContinuousAdjustment_<品种>_20100101-20260930_ratio1.10_log_common_demo.parquet`。`ratio1.10` 来自上游主力选择，`log_common` 表示全区间 log 共同偏移。文件名日期是请求范围，实际分钟起点以 metadata 为准。无主力偏移的时点保留原价和数量、复权值为空。

写入临时文件并复读全部内容后安装；生成身份与实际内容一致时保留已有文件。内容摘要使用固定行数的 Arrow batch，与月份和 Parquet 分组无关，避免将全历史同时装入内存。metadata 沿用 `demo_identity`，不新增外置说明文件或版本后缀。


In [16]:
import platform

method_notebook_path = candidate_root / "04_Research/a01_Methods/b02_MainContinuousAdjustment/c01_MainContinuousAdjustment.ipynb"
method_notebook = json.loads(method_notebook_path.read_text(encoding="utf-8"))
method_dependency_source = "\n\n".join("".join(cell["source"]) for cell in method_notebook["cells"]
    if cell["cell_type"] == "code" and {"export", "demo-dependency"}.issubset(cell.get("metadata", {}).get("tags", [])))
adjustment_demo_identity = {
    "artifact_role": "method_demo",
    "producer_notebook": method_notebook_path.relative_to(candidate_root).as_posix(),
    "method_version": MAIN_CONTINUOUS_ADJUSTMENT_VERSION,
    "method_source_tag": "demo-dependency",
    "method_source_sha256": hashlib.sha256(method_dependency_source.encode()).hexdigest(),
    "dependency_source_sha256": {
        "notebook_loader": hashlib.sha256((candidate_root / "04_Research/a00_notebook_loader.py").read_bytes()).hexdigest(),
        "data_contracts": hashlib.sha256((candidate_root / "config/data_contracts.py").read_bytes()).hexdigest(),
    },
    "exchange_code": demo_exchange_code, "underlying_code": demo_underlying_code,
    "scope": demo_scope, "timezone": ADJUSTMENT_TIMEZONE, "interval_closed": "both",
    "actual_bar_at_start": str(actual_bar_at_start),
    "actual_bar_at_end": str(actual_bar_at_end),
    "upstream_demo_identity": verified_upstream_identity,
    "input_identity": minute_input_identity,
    "parameters": {
        "price_transform": "natural_log_positive_finite; raw zero preserved; zero/null derived prices null; negative rejected",
        "gap": "new segment first log_open - old segment last log_close",
        "backward_shift": "within-train reverse cumulative gaps; test zero",
        "test_start_date": demo_test_start_date.isoformat(),
        "forward_shift": "train zero; test negative cumulative within-test past gaps, first segment zero",
        "linear_deduction": "train linspace(G,0,N_train); test zero",
        "endpoint_policy": "train first/last and test first observed main minute positive OHLC unchanged; no split-boundary compensation",
        "common_offset": "B_train+F_test-D_train; broadcast by trading_date,bar_at",
        "zero_offset_prices": "copy positive original exactly; zero/null derived prices null",
        "missing_policy": "report all theoretical-grid gaps including segment edges; anchors use first/last observed minute; reject empty segments/unavailable anchor logs/unresolved days; no filling",
        "no_main_offset": "adjusted fields null; original/log/quantities retained",
        "quantity_policy": "original per-contract; main uses selected contract; no variety aggregate",
    },
    "runtime": {"python": sys.version.split()[0], "python_executable": sys.executable,
        "pandas": pd.__version__, "pyarrow": pa.__version__, "numpy": np.__version__, "platform": platform.platform()},
    "diagnostic_summary": {
        "contracts": len(all_contract_codes),
        "trading_days": int(main_segment_daily_df["trading_date"].nunique()),
        "main_minutes": len(adjusted_main_minute_df), "segments": len(main_segment_summary_df),
        "training_minutes": adjustment_diagnostics["training_minute_count"],
        "test_minutes": adjustment_diagnostics["test_minute_count"],
        "rolls": len(roll_diagnostics_df), "missing_main_minutes": len(missing_main_minute_df),
        "missing_main_trading_dates": int(missing_main_minute_df["trading_date"].nunique()),
        "no_main_offset_times": len(unsupported_times_df),
        "original_ohlc_order_anomaly_rows": anomaly_rows,
        "total_log_gap": adjustment_diagnostics["total_log_gap"],
        "interval_count": adjustment_diagnostics["interval_count"],
        "per_interval_log_gap": adjustment_diagnostics["per_interval_log_gap"],
    },
}

demo_training_end_date = (pd.Timestamp(demo_test_start_date) - pd.Timedelta(days=1)).date()
demo_range_token = (
    f"train{demo_mapping_read_start_date:%Y%m%d}-{demo_training_end_date:%Y%m%d}_"
    f"test{demo_test_start_date:%Y%m%d}-{demo_mapping_read_end_date:%Y%m%d}")
adjustment_demo_path = method_notebook_path.parent / "c01_MainContinuousAdjustment_DemoData" / (
    f"{method_notebook_path.stem}_{demo_underlying_code}_{demo_range_token}_"
    f"ratio{demo_ratio_token}_log_common_demo.parquet")


def adjusted_month_batches():
    """逐批消费完整区间已确定的偏移，并核对每批变换性质。"""
    for month_number, (period, (input_filter, expected_hash)) in enumerate(month_inputs.items(), 1):
        table = minute_dataset.to_table(filter=input_filter, columns=FUTURES_MINUTE_SCHEMA.names)
        if arrow_content_sha256(table, FUTURES_MINUTE_SCHEMA, minute_sort_keys) != expected_hash:
            raise RuntimeError(f"{period} 行情在两遍读取间改变，请重新运行")
        month = pd.Period(period, freq='M')
        daily = main_segment_daily_df.loc[main_segment_daily_df['trading_date'].between(month.start_time.date(), month.end_time.date())]
        all_log_df, _, _diagnostics = prepare_main_contract_minutes(arrow_to_pandas(table, FUTURES_MINUTE_SCHEMA), daily, month_calendars[period])
        offsets = minute_log_offset_df.loc[minute_log_offset_df['trading_date'].between(month.start_time.date(), month.end_time.date())]
        adjusted_df = apply_minute_log_offsets(all_log_df, offsets)
        assert len(adjusted_df) == len(all_log_df)
        assert adjusted_df[ALL_CONTRACT_LOG_MINUTE_SCHEMA.names].equals(all_log_df)
        expected_main = adjusted_main_minute_df.loc[adjusted_main_minute_df['trading_date'].between(month.start_time.date(), month.end_time.date())].reset_index(drop=True)
        assert adjusted_df.loc[adjusted_df['is_main']].reset_index(drop=True).equals(expected_main)
        shifts = adjusted_df['log_adjustment'].to_numpy(dtype=float, na_value=np.nan)
        same_time = adjusted_df['bar_at'].eq(adjusted_df['bar_at'].shift()).fillna(False).to_numpy(dtype=bool)[1:]
        zero_shift = shifts == 0
        for name in OHLC_FIELDS:
            raw_log = adjusted_df[f'log_{name}'].to_numpy(dtype=float, na_value=np.nan)
            adj_log = adjusted_df[f'adjusted_log_{name}'].to_numpy(dtype=float, na_value=np.nan)
            np.testing.assert_allclose(adj_log, raw_log + shifts, atol=2e-14, rtol=1e-10, equal_nan=True)
            valid_pairs = same_time & np.isfinite(shifts[:-1]) & np.isfinite(shifts[1:])
            np.testing.assert_allclose(np.diff(adj_log)[valid_pairs], np.diff(raw_log)[valid_pairs], atol=2e-14, rtol=1e-10, equal_nan=True)
            positive_zero_shift = zero_shift & adjusted_df[f'raw_{name}'].gt(0).fillna(False).to_numpy(dtype=bool)
            assert adjusted_df.loc[positive_zero_shift, f'adjusted_{name}'].equals(adjusted_df.loc[positive_zero_shift, f'raw_{name}'])
            zero_price = adjusted_df[f'raw_{name}'].eq(0).fillna(False)
            assert adjusted_df.loc[zero_price, [f'log_{name}', f'adjusted_log_{name}', f'adjusted_{name}']].isna().all().all()
        assert adjusted_df.loc[adjusted_df['log_adjustment'].isna(), [f'adjusted_{name}' for name in OHLC_FIELDS]].isna().all().all()
        yield adjusted_df
        if month_number == 1 or month_number % 12 == 0 or month_number == len(month_inputs):
            print(f"{demo_underlying_code} 保存 {month_number}/{len(month_inputs)} 月；总耗时 {perf_counter()-demo_started:.1f} 秒", flush=True)


adjustment_demo_path.parent.mkdir(parents=True, exist_ok=True)
adjustment_demo_save = save_main_adjustment_demo(adjusted_month_batches(), demo_path=adjustment_demo_path, demo_identity=adjustment_demo_identity)
print('保存并复读验收：', adjustment_demo_path)
print('行数：', adjustment_demo_save['identity']['output_rows'], 'MiB：', round(adjustment_demo_save['file_bytes']/2**20, 2))
print('保留已有字节：', adjustment_demo_save['preserved_existing'])
print('SHA-256：', adjustment_demo_save['identity']['output_content_sha256'])
saved_main_df = pq.read_table(adjustment_demo_path, filters=[('is_main', '=', True)]).to_pandas(types_mapper=pd.ArrowDtype)
assert saved_main_df.equals(adjusted_main_minute_df)
display(saved_main_df.head(3).rename(columns=ADJUSTMENT_COLUMN_LABELS))
display(saved_main_df.tail(3).rename(columns=ADJUSTMENT_COLUMN_LABELS))

RB 保存 1/201 月；总耗时 69.1 秒


RB 保存 12/201 月；总耗时 73.0 秒


RB 保存 24/201 月；总耗时 77.5 秒


RB 保存 36/201 月；总耗时 82.0 秒


RB 保存 48/201 月；总耗时 86.2 秒


...

RB 保存 192/201 月；总耗时 155.2 秒


RB 保存 201/201 月；总耗时 159.1 秒


保存并复读验收： E:\Latitude_Analytics_v2\04_Research\a01_Methods\b02_MainContinuousAdjustment\c02_DemoData\c01_MainContinuousAdjustment_RB_20100101-20260930_ratio1.10_log_common_demo.parquet
行数： 15344910 MiB： 1102.91
保留已有字节： False
SHA-256： 3c44fbf17cbe067db99ef8819fdc0b4921c3b8a752af2089db176d64fd1ccfe6


,contract_code（项目标准固定月份合约代码）,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,trading_date（bar 归属的交易日，而非其自然日期）,session_number（bar 所属 Session 编号）,bar_at（一分钟 bar 的结束时刻）,raw_open（原始本分钟开盘价）,raw_high（原始本分钟最高价）,raw_low（原始本分钟最低价）,raw_close（原始本分钟收盘价）,volume（本分钟成交量）,money（本分钟成交金额）,open_interest（本分钟结束时持仓量）,source（行情来源与关键参数）,updated_at（本行最后写入时间）,year（归属交易年份）,month（归属交易月份）,log_open（原始本分钟开盘价自然对数）,log_high（原始本分钟最高价自然对数）,log_low（原始本分钟最低价自然对数）,log_close（原始本分钟收盘价自然对数）,main_contract_code（当日主力合约）,segment_id（连续主力片段编号）,is_main（是否当日主力）,backward_log_shift（训练累计后续换约对数偏移）,forward_log_shift（测试累计过去换约对数偏移）,linear_log_deduction（线性扣除的对数偏移）,log_adjustment（共同复权对数偏移）,adjustment_status（复权状态）,adjusted_log_open（复权本分钟开盘价自然对数）,adjusted_log_high（复权本分钟最高价自然对数）,adjusted_log_low（复权本分钟最低价自然对数）,adjusted_log_close（复权本分钟收盘价自然对数）,adjusted_open（复权本分钟开盘价）,adjusted_high（复权本分钟最高价）,adjusted_low（复权本分钟最低价）,adjusted_close（复权本分钟收盘价）
0,RB1005.XSGE,XSGE,RB,2010-01-04,1,2010-01-04 09:01:00+08:00,4523.0,4540.0,4523.0,4535.0,21988.0,996915100.0,914162.0,JQData_get_price_1m_skip_paused,2026-08-02 13:53:23+00:00,2010,1,8.416931,8.420682,8.416931,8.41958,RB1005.XSGE,1,True,-0.307746,0.0,-0.307746,0.0,adjusted,8.416931,8.420682,8.416931,8.41958,4523.0,4540.0,4523.0,4535.0
1,RB1005.XSGE,XSGE,RB,2010-01-04,1,2010-01-04 09:02:00+08:00,4535.0,4535.0,4524.0,4528.0,16676.0,755024100.0,913964.0,JQData_get_price_1m_skip_paused,2026-08-02 13:53:23+00:00,2010,1,8.41958,8.41958,8.417152,8.418036,RB1005.XSGE,1,True,-0.307746,0.0,-0.307746,-0.0,adjusted,8.41958,8.41958,8.417152,8.418035,4534.998769,4534.998769,4523.998772,4527.998771
2,RB1005.XSGE,XSGE,RB,2010-01-04,1,2010-01-04 09:03:00+08:00,4528.0,4540.0,4528.0,4538.0,15176.0,688346280.0,914710.0,JQData_get_price_1m_skip_paused,2026-08-02 13:53:23+00:00,2010,1,8.418036,8.420682,8.418036,8.420242,RB1005.XSGE,1,True,-0.307746,0.0,-0.307745,-0.000001,adjusted,8.418035,8.420682,8.418035,8.420241,4527.997543,4539.997536,4527.997543,4537.997537


,contract_code（项目标准固定月份合约代码）,exchange_code（合约所属交易所）,underlying_code（合约所属品种）,trading_date（bar 归属的交易日，而非其自然日期）,session_number（bar 所属 Session 编号）,bar_at（一分钟 bar 的结束时刻）,raw_open（原始本分钟开盘价）,raw_high（原始本分钟最高价）,raw_low（原始本分钟最低价）,raw_close（原始本分钟收盘价）,volume（本分钟成交量）,money（本分钟成交金额）,open_interest（本分钟结束时持仓量）,source（行情来源与关键参数）,updated_at（本行最后写入时间）,year（归属交易年份）,month（归属交易月份）,log_open（原始本分钟开盘价自然对数）,log_high（原始本分钟最高价自然对数）,log_low（原始本分钟最低价自然对数）,log_close（原始本分钟收盘价自然对数）,main_contract_code（当日主力合约）,segment_id（连续主力片段编号）,is_main（是否当日主力）,backward_log_shift（训练累计后续换约对数偏移）,forward_log_shift（测试累计过去换约对数偏移）,linear_log_deduction（线性扣除的对数偏移）,log_adjustment（共同复权对数偏移）,adjustment_status（复权状态）,adjusted_log_open（复权本分钟开盘价自然对数）,adjusted_log_high（复权本分钟最高价自然对数）,adjusted_log_low（复权本分钟最低价自然对数）,adjusted_log_close（复权本分钟收盘价自然对数）,adjusted_open（复权本分钟开盘价）,adjusted_high（复权本分钟最高价）,adjusted_low（复权本分钟最低价）,adjusted_close（复权本分钟收盘价）
1278912,RB2701.XSGE,XSGE,RB,2026-09-30,4,2026-09-30 14:58:00+08:00,3112.0,3113.0,3112.0,3112.0,1974.0,61439540.0,1595528.0,JQData_get_price_1m_skip_paused_fq_none,2026-10-02 08:04:14.866421+00:00,2026,9,8.043021,8.043342,8.043021,8.043021,RB2701.XSGE,51,True,0.0,-0.078652,0.0,-0.078652,adjusted,7.964369,7.96469,7.964369,7.964369,2876.612728,2877.53709,2876.612728,2876.612728
1278913,RB2701.XSGE,XSGE,RB,2026-09-30,4,2026-09-30 14:59:00+08:00,3113.0,3113.0,3112.0,3113.0,3899.0,121364160.0,1594314.0,JQData_get_price_1m_skip_paused_fq_none,2026-10-02 08:04:14.866421+00:00,2026,9,8.043342,8.043342,8.043021,8.043342,RB2701.XSGE,51,True,0.0,-0.078652,0.0,-0.078652,adjusted,7.96469,7.96469,7.964369,7.96469,2877.53709,2877.53709,2876.612728,2877.53709
1278914,RB2701.XSGE,XSGE,RB,2026-09-30,4,2026-09-30 15:00:00+08:00,3113.0,3113.0,3112.0,3112.0,5452.0,169692770.0,1592266.0,JQData_get_price_1m_skip_paused_fq_none,2026-10-02 08:04:14.866421+00:00,2026,9,8.043342,8.043342,8.043021,8.043021,RB2701.XSGE,51,True,0.0,-0.078652,0.0,-0.078652,adjusted,7.96469,7.96469,7.964369,7.964369,2877.53709,2877.53709,2876.612728,2876.612728
